<a href="https://colab.research.google.com/github/isocan/ML-accelerated-ORR/blob/main/notebooks/03_stageII_AQCat25_adsorbml_relaxation_CHE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/isocan/ML-accelerated-ORR/blob/main/notebooks/03_stageII_aqcat25_ev2_spin_aware_adsorbml_relaxation.ipynb"
   target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg"
       alt="Open in Colab"/>
</a>

# Stage II — AQCat25-EV2 Spin-Aware AdsorbML Screening

This Google Colab notebook is the second Stage II reproducibility tutorial in the
multi-fidelity oxygen reduction reaction workflow. The complete study applied the
AQCat25-EV2 procedure to the same 24 top candidates selected after Stage I. For a
clear, executable demonstration, this public notebook runs one selected slab:

```python
SURFACE_ID = "mp-10260_111"
```

## Scientific workflow

1. Install the AQCat25-compatible PyTorch and PyG environment.
2. Restart the Colab kernel once to avoid binary ABI conflicts.
3. Download the gated AQCat25-EV2 checkpoint and patched source files.
4. Clone FairChem `fairchem_core-1.10.0`, apply the AQCat patches, and install it.
5. Load the AQCat25-EV2 calculator once in high-fidelity, spin-on mode.
6. Generate and relax the bare slab first.
7. Rebuild the FairChem slab object from the AQCat-relaxed bare geometry.
8. Generate heuristic and random O*, OH*, and OOH* configurations.
9. Relax every adsorbate-slab configuration with AQCat25-EV2.
10. Validate trajectories, classify adsorption sites, and select unique-site and
    global minima.
11. Export the relaxed bare slab and global O*, OH*, and OOH* minima as
    spin-aware, high-fidelity RPBE single-point VASP calculations.

## Energy interpretation

AQCat25-EV2 is an adsorption-energy machine-learning potential. For an
adsorbate-slab structure, the model energy returned after relaxation is stored
directly as

$$
E_{\mathrm{ads}}^{\mathrm{AQCat}}
= E_{\mathrm{model}}(\mathrm{adsorbate+slab}).
$$

The principal result column is `AQCat_pred_ads_energy_eV`.

The bare-slab model energy is retained only as a diagnostic quantity needed to
reproduce the geometry optimization. It is **not subtracted** from the AQCat
adsorption-energy prediction, and no separate H2/H2O CHE reference calculation
is performed in this notebook.

## Spin and fidelity context

Every AQCat inference explicitly uses

```python
is_spin_off = False
is_low_fi = False
```

which requests the high-fidelity, spin-on FiLM context. The model card reports
that the patched calculator defaults to high fidelity and detects systems where
spin treatment is relevant; the explicit flags are retained here for transparent
reproducibility.

## VASP validation protocol

The exported folders are prepared for RPBE single-point calculations matching
the high-fidelity AQCat25 branch as closely as possible:

- `ENCUT = 500 eV`
- `PREC = Accurate`
- Gaussian smearing with `ISMEAR = 0` and `SIGMA = 0.1 eV`
- `ISPIN = 2` for systems containing AQCat25 magnetic elements
- Pymatgen-style initial magnetic moments
- no ionic update: `IBRION = -1`, `NSW = 1`

The in-plane Monkhorst-Pack mesh follows

$$
N_a=\max\left(1,\operatorname{round}\frac{40}{|\mathbf a|}\right),\qquad
N_b=\max\left(1,\operatorname{round}\frac{40}{|\mathbf b|}\right),\qquad
N_c=1.
$$

Licensed VASP PAW datasets are not redistributed. Each calculation directory
contains a simple non-runnable `POTCAR` placeholder.


## 1. Install the Colab environment

Run this cell once in a fresh GPU runtime. Do not import NumPy, SciPy, ASE,
PyTorch, PyG, or FairChem before the restart cell.

AQCat25-EV2 was released for PyTorch 2.4.0 with CUDA 12.1-compatible PyG wheels.
Replacing these binary packages inside an active Colab kernel can produce ABI
errors or a kernel crash, so the notebook deliberately separates installation
from imports.


In [1]:
from pathlib import Path
import subprocess
import sys

ENV_MARKER = Path("/content/.stageII_aqcat25_environment_installed")
RESTART_MARKER = Path("/content/.stageII_aqcat25_kernel_restarted")

if ENV_MARKER.exists():
    print("AQCat25 environment is already installed.")
else:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "uninstall",
            "-y",
            "torch",
            "torchvision",
            "torchaudio",
            "torch_geometric",
            "torch_scatter",
            "torch_sparse",
            "torch_cluster",
            "torch_spline_conv",
            "pyg_lib",
            "fairchem-core",
            "fairchem",
            "fairchem-data-oc",
        ],
        check=False,
    )

    commands = [
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--upgrade",
            "pip",
            "wheel",
            "setuptools<81",
            "packaging",
            "ninja",
            "cmake",
        ],
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--no-cache-dir",
            "numpy==1.26.4",
            "scipy==1.13.1",
            "pandas==2.2.3",
            "ase==3.23.0",
        ],
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--no-cache-dir",
            "torch==2.4.0",
            "--index-url",
            "https://download.pytorch.org/whl/cu121",
        ],
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--no-cache-dir",
            "pyg_lib",
            "torch_scatter",
            "torch_sparse",
            "torch_cluster",
            "torch_spline_conv",
            "-f",
            "https://data.pyg.org/whl/torch-2.4.0+cu121.html",
        ],
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--no-cache-dir",
            "torch_geometric==2.6.1",
            "e3nn",
            "submitit",
            "torchtnt",
            "hydra-core",
            "pymatgen",
            "orjson",
            "wandb",
            "tensorboard",
            "lmdb",
            "huggingface-hub>=0.30,<1",
            "numba",
            "datasets",
            "tqdm>=4.66",
            "requests",
            "gitpython",
            "py3Dmol>=2.4,<3",
            "ipywidgets>=8,<9",
        ],
    ]

    for command in commands:
        print("\nRunning:")
        print(" ".join(command))
        subprocess.check_call(command)

    ENV_MARKER.write_text("installed\n", encoding="utf-8")
    RESTART_MARKER.unlink(missing_ok=True)

    print("\nInstallation completed.")
    print("Run the next cell once to restart the Colab kernel.")



Running:
/usr/bin/python3 -m pip install --upgrade pip wheel setuptools<81 packaging ninja cmake

Running:
/usr/bin/python3 -m pip install --no-cache-dir numpy==1.26.4 scipy==1.13.1 pandas==2.2.3 ase==3.23.0

Running:
/usr/bin/python3 -m pip install --no-cache-dir torch==2.4.0 --index-url https://download.pytorch.org/whl/cu121

Running:
/usr/bin/python3 -m pip install --no-cache-dir pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.4.0+cu121.html

Running:
/usr/bin/python3 -m pip install --no-cache-dir torch_geometric==2.6.1 e3nn submitit torchtnt hydra-core pymatgen orjson wandb tensorboard lmdb huggingface-hub>=0.30,<1 numba datasets tqdm>=4.66 requests gitpython py3Dmol>=2.4,<3 ipywidgets>=8,<9

Installation completed.
Run the next cell once to restart the Colab kernel.


In [ ]:
# Required one-time restart after replacing the binary scientific stack.

from pathlib import Path
import os
import time

ENV_MARKER = Path("/content/.stageII_aqcat25_environment_installed")
RESTART_MARKER = Path("/content/.stageII_aqcat25_kernel_restarted")

if not ENV_MARKER.exists():
    raise RuntimeError(
        "The environment installation did not finish. Run the previous cell "
        "and inspect the complete pip output."
    )

if RESTART_MARKER.exists():
    print("Kernel restart already completed. Continue below.")
else:
    RESTART_MARKER.write_text("restarted\n", encoding="utf-8")
    print("Restarting the Colab kernel now...")
    time.sleep(1)
    os.kill(os.getpid(), 9)


Restarting the Colab kernel now...


In [1]:
# Verify only after the kernel restart.

import sys

import ase
import numpy as np
import pandas as pd
import scipy
import torch

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("pandas:", pd.__version__)
print("ASE:", ase.__version__)

assert torch.__version__.split("+")[0] == "2.4.0"
assert np.__version__ == "1.26.4"
assert scipy.__version__ == "1.13.1"
assert pd.__version__ == "2.2.3"
assert ase.__version__ == "3.23.0"

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable. Select Runtime -> Change runtime type -> T4 GPU."
    )

print("\nBinary environment verification passed.")


Python: 3.12.13
PyTorch: 2.4.0+cu121
CUDA build: 12.1
CUDA available: True
GPU: Tesla T4
NumPy: 1.26.4
SciPy: 1.13.1
pandas: 2.2.3
ASE: 3.23.0

Binary environment verification passed.


## 2. Authenticate and download AQCat25-EV2

The model repository is gated and licensed for non-commercial use. Accept the
repository conditions on Hugging Face, create a read token, and save it in
Colab as a secret named `HF_TOKEN`. The interactive login is used as a fallback.


In [2]:
import os
from huggingface_hub import login, notebook_login

hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN")

if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    os.environ["HF_TOKEN"] = hf_token
    print("Authenticated using the HF_TOKEN Colab secret.")
else:
    print("No HF_TOKEN secret found. Complete the interactive login.")
    notebook_login()


Authenticated using the HF_TOKEN Colab secret.


In [3]:
from pathlib import Path
from huggingface_hub import snapshot_download

AQCAT_DIR = Path("/content/aqcat25-ev2")

snapshot_download(
    repo_id="SandboxAQ/aqcat25-ev2",
    repo_type="model",
    allow_patterns=[
        "checkpoints_aqcat_ev2/*",
        "ev2_film/*",
        "patched_code/*",
    ],
    local_dir=str(AQCAT_DIR),
    token=os.environ.get("HF_TOKEN"),
)

CHECKPOINT_PATH = (
    AQCAT_DIR
    / "checkpoints_aqcat_ev2"
    / "ev2-in+midFiLM-AQCat25+OC20-20M_20251008_223220.pt"
)
PATCHED_MODEL_SOURCE = (
    AQCAT_DIR / "ev2_film" / "equiformer_v2_film.py"
)
PATCHED_ASE_SOURCE = (
    AQCAT_DIR / "patched_code" / "ase_utils.py"
)

for path in (
    CHECKPOINT_PATH,
    PATCHED_MODEL_SOURCE,
    PATCHED_ASE_SOURCE,
):
    print(path, "->", path.exists())
    if not path.exists():
        raise FileNotFoundError(path)


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

checkpoints_aqcat_ev2/ev2-inFiLM-OC20-ft(…):   0%|          | 0.00/126M [00:00<?, ?B/s]

ase_utils.py: 0.00B [00:00, ?B/s]

checkpoints_aqcat_ev2/ev2-153M-OC20-ft-A(…):   0%|          | 0.00/620M [00:00<?, ?B/s]

checkpoints_aqcat_ev2/ev2-OC20-ft-AQCat2(…):   0%|          | 0.00/126M [00:00<?, ?B/s]

checkpoints_aqcat_ev2/ev2-in+midFiLM-AQC(…):   0%|          | 0.00/126M [00:00<?, ?B/s]

equiformer_v2_film.py: 0.00B [00:00, ?B/s]

/content/aqcat25-ev2/checkpoints_aqcat_ev2/ev2-in+midFiLM-AQCat25+OC20-20M_20251008_223220.pt -> True
/content/aqcat25-ev2/ev2_film/equiformer_v2_film.py -> True
/content/aqcat25-ev2/patched_code/ase_utils.py -> True


## 3. Patch and install FairChem 1.10.0

AQCat25-EV2 requires two files supplied by the model repository. This cell
clones the exact FairChem release used by AQCat, copies the FiLM model and
patched ASE calculator into the source tree, and installs that patched source
without changing the binary environment again.


In [4]:
import importlib
import shutil
import subprocess
import sys
from pathlib import Path

FAIRCHEM_DIR = Path("/content/fairchem-aqcat25")
FAIRCHEM_PACKAGE_DIR = FAIRCHEM_DIR / "packages" / "fairchem-core"
FAIRCHEM_SRC_DIR = FAIRCHEM_PACKAGE_DIR / "src"

if FAIRCHEM_DIR.exists():
    shutil.rmtree(FAIRCHEM_DIR)

subprocess.run(
    [
        "git",
        "clone",
        "--branch",
        "fairchem_core-1.10.0",
        "--depth",
        "1",
        "https://github.com/facebookresearch/fairchem.git",
        str(FAIRCHEM_DIR),
    ],
    check=True,
)

model_destination = (
    FAIRCHEM_SRC_DIR
    / "fairchem"
    / "core"
    / "models"
    / "equiformer_v2"
    / "equiformer_v2_film.py"
)
ase_destination = (
    FAIRCHEM_SRC_DIR
    / "fairchem"
    / "core"
    / "common"
    / "relaxation"
    / "ase_utils.py"
)

shutil.copy2(PATCHED_MODEL_SOURCE, model_destination)
shutil.copy2(PATCHED_ASE_SOURCE, ase_destination)

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-e",
        str(FAIRCHEM_PACKAGE_DIR),
        "--no-deps",
    ],
    check=True,
)

src_text = str(FAIRCHEM_SRC_DIR)
if src_text not in sys.path:
    sys.path.insert(0, src_text)

for module_name in list(sys.modules):
    if module_name == "fairchem" or module_name.startswith("fairchem."):
        del sys.modules[module_name]

importlib.invalidate_caches()

print("Patched FairChem source:", FAIRCHEM_SRC_DIR)
print("FiLM patch:", model_destination.exists())
print("ASE calculator patch:", ase_destination.exists())


Patched FairChem source: /content/fairchem-aqcat25/packages/fairchem-core/src
FiLM patch: True
ASE calculator patch: True


## 4. Imports and calculation settings

The notebook demonstrates one Stage II candidate. Increase
`RANDOM_SITES_PER_ADSORBATE` or change `SURFACE_ID` to reproduce additional
candidates. The calculator is loaded once and reused to avoid repeated GPU
allocation.


In [5]:
from __future__ import annotations

import copy
import io
import itertools
import json
import random
import re
import shutil
import textwrap
import time
import warnings
from pathlib import Path

import ase.io as aseio
import ipywidgets as widgets
import numpy as np
import pandas as pd
import py3Dmol
import torch

from ase import Atoms
from ase.calculators.singlepoint import SinglePointCalculator
from ase.constraints import FixAtoms
from ase.data import atomic_numbers, covalent_radii
from ase.optimize import LBFGS
from IPython.display import clear_output, display
from tqdm.auto import tqdm

from fairchem.core.common.relaxation.ase_utils import patched_calc
from fairchem.data.oc.core import Adsorbate, AdsorbateSlabConfig, Bulk, Slab
from fairchem.data.oc.utils import DetectTrajAnomaly

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning, module="ase")

# Candidate selection
SURFACE_ID = "mp-10260_111"
SURFACE_INDEX = 0

# AQCat25-EV2 inference
MODEL_NAME = "AQCat25-EV2-in+midFiLM-AQCat25+OC20-20M"
SPIN_CONTEXT = "spin_on"
IS_SPIN_OFF = False
IS_LOW_FIDELITY = False
ADSORBATE_NAMES = ("O", "OH", "OOH")
ADSORBATE_SIZES = {"O": 1, "OH": 2, "OOH": 3}
RANDOM_SITES_PER_ADSORBATE = 20
FMAX_THRESHOLD = 0.05
MAX_RELAX_STEPS = 400
OPTIMIZER_MAXSTEP_A = 0.10
MIN_INTERATOMIC_DISTANCE_A = 0.70
OVERWRITE_EXISTING = False

# AQCat25 RPBE single-point validation protocol
KPOINT_DENSITY = 40.0
VASP_RUN_MODE = "Single Point"
VASP_PREC = "Accurate"
VASP_ENCUT_EV = 500
VASP_EDIFF = 1e-4
VASP_NELM = 250
VASP_SIGMA_EV = 0.1
VASP_SYMPREC = 1e-10
VASP_NPAR = 4
VASP_ISTART = 0  # No WAVECAR is distributed with the reproducibility archive.

# AQCat25 Table 6 initial magnetic moments, in μB.
# Slab atoms not listed here receive 0.00 μB.
# Adsorbate O and H always receive 0.00 μB.
AQCAT_INITIAL_MAGMOM = {
    "V": 5.00,
    "Cr": 5.00,
    "Mn": 5.00,
    "Fe": 5.00,
    "Co": 5.00,
    "Ni": 5.00,
    "Cu": 1.73,
    "Mo": 5.00,
    "Ru": 2.20,
    "W": 5.00,
    "Os": 2.20,
    "Ce": 5.00,
}
AQCAT_SPIN_ELEMENTS = set(AQCAT_INITIAL_MAGMOM)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

ROOT_DIR = Path("/content/stageII_aqcat25_ev2_results")
ROOT_DIR.mkdir(parents=True, exist_ok=True)

print("Selected surface:", SURFACE_ID)
print("Model:", MODEL_NAME)
print("AQCat context: high fidelity, spin on")
print("Force threshold:", FMAX_THRESHOLD, "eV/Å")


Selected surface: mp-10260_111
Model: AQCat25-EV2-in+midFiLM-AQCat25+OC20-20M
AQCat context: high fidelity, spin on
Force threshold: 0.05 eV/Å


In [6]:
selected_surface = SURFACE_ID.strip()


def parse_surface_id(surface_id: str) -> tuple[str, tuple[int, int, int]]:
    """Parse a surface identifier such as mp-10260_111."""
    try:
        bulk_id, miller_text = surface_id.rsplit("_", 1)
    except ValueError as exc:
        raise ValueError(
            "Use the form 'mp-id_facet', for example 'mp-10260_111'."
        ) from exc

    if not bulk_id.startswith("mp-"):
        raise ValueError(f"Invalid Materials Project ID: {bulk_id}")
    if len(miller_text) != 3 or not miller_text.isdigit():
        raise ValueError(
            "Use a three-digit positive facet suffix such as 111, 110, or 100."
        )

    return bulk_id, tuple(int(value) for value in miller_text)


BULK_ID, MILLER_INDEX = parse_surface_id(selected_surface)
SYSTEM_ID = (
    f"{BULK_ID}_{''.join(map(str, MILLER_INDEX))}"
    f"_term{SURFACE_INDEX}_aqcat25_spin_on"
)

SYSTEM_DIR = ROOT_DIR / SYSTEM_ID
TRAJECTORY_DIR = SYSTEM_DIR / "trajectories"
LOG_DIR = SYSTEM_DIR / "logs"
POSCAR_DIR = SYSTEM_DIR / "poscars"
TABLE_DIR = SYSTEM_DIR / "tables"
VASP_DIR = SYSTEM_DIR / "vasp"

for directory in (
    SYSTEM_DIR,
    TRAJECTORY_DIR,
    LOG_DIR,
    POSCAR_DIR,
    TABLE_DIR,
    VASP_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)

print("Materials Project bulk ID:", BULK_ID)
print("Requested Miller index:", MILLER_INDEX)
print("Selected termination:", SURFACE_INDEX)
print("Output directory:", SYSTEM_DIR)


Materials Project bulk ID: mp-10260
Requested Miller index: (1, 1, 1)
Selected termination: 0
Output directory: /content/stageII_aqcat25_ev2_results/mp-10260_111_term0_aqcat25_spin_on


## 5. Load the AQCat25-EV2 calculator once

The patched calculator is instantiated once on the GPU and reused for every
bare-slab and adsorbate-slab relaxation. Resetting its ASE cache between
structures prevents results from a previous structure or FiLM context from
being reused.


In [7]:
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(CHECKPOINT_PATH)
if not torch.cuda.is_available():
    raise RuntimeError("AQCat25-EV2 requires a CUDA-enabled Colab runtime.")

AQCAT_CALC = patched_calc(
    checkpoint_path=str(CHECKPOINT_PATH),
    cpu=False,
)

print("Checkpoint:", CHECKPOINT_PATH)
print("GPU:", torch.cuda.get_device_name(0))
print("AQCat25-EV2 calculator loaded once.")


INFO:root:local rank base: 0
INFO:root:amp: false
cmd:
  checkpoint_dir: /content/checkpoints/2026-07-30-16-38-24
  commit: core:977a803,experimental:NA
  identifier: ''
  logs_dir: /content/logs/wandb/2026-07-30-16-38-24
  print_every: 100
  results_dir: /content/results/2026-07-30-16-38-24
  seed: null
  timestamp_id: 2026-07-30-16-38-24
  version: 1.10.0
dataset:
  a2g_args:
    r_data_keys:
    - is_spin_off
    - is_low_fi
    - adsorption_energy
    r_energy: false
    r_forces: true
  format: ase_db
  key_mapping:
    adsorption_energy: energy
  seed: 1
  transforms:
    normalizer:
      energy:
        mean: -0.7554450631141663
        stdev: 2.887317180633545
      forces:
        mean: 0
        stdev: 2.887317180633545
evaluation_metrics:
  metrics:
    energy:
    - mae
    forces:
    - mae
    - cosine_similarity
    - magnitude_error
    misc:
    - energy_forces_within_threshold
  primary_metric: forces_mae
gp_gpus: null
gpus: 1
logger: wandb
loss_functions:
- energy:


Checkpoint: /content/aqcat25-ev2/checkpoints_aqcat_ev2/ev2-in+midFiLM-AQCat25+OC20-20M_20251008_223220.pt
GPU: Tesla T4
AQCat25-EV2 calculator loaded once.


## 6. Generate the selected slab

FairChem reconstructs the Materials Project bulk and enumerates the requested
surface termination. OC tags and fixed-atom constraints are preserved. The bare
slab is relaxed before any adsorbate is placed.


In [8]:
def rebuild_atoms(
    atoms: Atoms,
    *,
    slab: bool = False,
    molecule: bool = False,
) -> Atoms:
    """Recreate ASE atoms while preserving geometry, tags, and cell."""
    tags = np.asarray(atoms.get_tags(), dtype=int)
    rebuilt = Atoms(
        numbers=np.asarray(atoms.get_atomic_numbers(), dtype=int),
        positions=np.asarray(atoms.get_positions(), dtype=float),
        cell=np.asarray(atoms.cell.array, dtype=float),
        pbc=False if molecule else np.asarray(atoms.get_pbc(), dtype=bool),
        tags=tags,
    )
    if slab:
        rebuilt.set_pbc(True)
        fixed = np.flatnonzero(tags == 0)
        if len(fixed):
            rebuilt.set_constraint(FixAtoms(indices=fixed))
    return rebuilt


def ensure_finite_geometry(atoms: Atoms, label: str) -> None:
    if not np.isfinite(atoms.positions).all():
        raise ValueError(f"Non-finite positions in {label}.")
    if not np.isfinite(atoms.cell.array).all():
        raise ValueError(f"Non-finite cell in {label}.")


def has_bad_contact(
    atoms: Atoms,
    cutoff: float = MIN_INTERATOMIC_DISTANCE_A,
) -> bool:
    distances = atoms.get_all_distances(mic=True)
    np.fill_diagonal(distances, np.inf)
    return bool(float(np.nanmin(distances)) < cutoff)


bulk = Bulk(bulk_src_id_from_db=BULK_ID)
if getattr(bulk, "src_id", BULK_ID) != BULK_ID:
    raise RuntimeError(
        f"Requested {BULK_ID}, but the database returned "
        f"{getattr(bulk, 'src_id', None)}."
    )

bulk.atoms = rebuild_atoms(bulk.atoms)
generated = Slab.from_bulk_get_specific_millers(
    bulk=bulk,
    specific_millers=MILLER_INDEX,
)

if isinstance(generated, Slab):
    slab_candidates = [generated]
elif generated is None:
    slab_candidates = []
else:
    slab_candidates = list(generated)

if not slab_candidates:
    raise RuntimeError(f"No {MILLER_INDEX} slab was generated for {BULK_ID}.")
if not 0 <= SURFACE_INDEX < len(slab_candidates):
    raise IndexError("SURFACE_INDEX is outside the generated termination range.")

raw_slab = slab_candidates[SURFACE_INDEX]
clean_initial = rebuild_atoms(raw_slab.atoms, slab=True)
ensure_finite_geometry(clean_initial, "clean slab")

clean_initial_path = SYSTEM_DIR / "clean_slab_initial.traj"
aseio.write(clean_initial_path, clean_initial)

print("Bulk formula:", bulk.atoms.get_chemical_formula())
print("Slab formula:", clean_initial.get_chemical_formula())
print("Generated terminations:", len(slab_candidates))
print("Selected termination:", SURFACE_INDEX)
display(
    pd.Series(clean_initial.get_tags())
    .value_counts()
    .sort_index()
    .rename_axis("OC_tag")
    .reset_index(name="count")
)


Bulk formula: Ni3Sb
Slab formula: Ni36Sb12
Generated terminations: 4
Selected termination: 0


,OC_tag,count
0,0,36
1,1,12


## 7. Construct O*, OH*, and OOH*

FairChem database adsorbates are used when available. The explicit fallback
structures make the notebook robust to small database differences between
FairChem releases.


In [9]:
def build_o() -> Atoms:
    return Atoms("O", positions=[[0.0, 0.0, 0.0]], pbc=False)


def build_oh() -> Atoms:
    return Atoms(
        ["O", "H"],
        positions=[[0.0, 0.0, 0.0], [0.97, 0.0, 0.0]],
        pbc=False,
    )


def build_ooh() -> Atoms:
    return Atoms(
        ["O", "O", "H"],
        positions=[
            [0.000, 0.000, 0.000],
            [0.000, 0.000, 1.450],
            [0.940, 0.000, 1.750],
        ],
        pbc=False,
    )


def load_or_build(label: str, fallback: Atoms, binding: list[int]) -> Adsorbate:
    try:
        adsorbate = Adsorbate(adsorbate_smiles_from_db=label)
        adsorbate.atoms = rebuild_atoms(adsorbate.atoms, molecule=True)
        return adsorbate
    except Exception:
        return Adsorbate(
            adsorbate_atoms=fallback,
            adsorbate_binding_indices=binding,
        )


adsorbates = {
    "O": load_or_build("*O", build_o(), [0]),
    "OH": load_or_build("*OH", build_oh(), [0]),
    "OOH": load_or_build("*OOH", build_ooh(), [0]),
}

adsorbate_summary = pd.DataFrame(
    [
        {
            "adsorbate": name,
            "formula": adsorbate.atoms.get_chemical_formula(),
            "n_atoms": len(adsorbate.atoms),
            "binding_indices": list(adsorbate.binding_indices),
        }
        for name, adsorbate in adsorbates.items()
    ]
)
display(adsorbate_summary)


,adsorbate,formula,n_atoms,binding_indices
0,O,O,1,[0]
1,OH,HO,2,[0]
2,OOH,HO2,3,[0]


## 8. Shared AQCat25-EV2 relaxation routine

The same calculator, force threshold, spin context, and high-fidelity context
are applied to the bare slab and all adsorbate-slab structures. Trajectories
store the final model energy and forces using an ASE single-point calculator so
they remain readable without reloading the neural network.


In [10]:
def maximum_force(forces: np.ndarray) -> float:
    forces = np.asarray(forces, dtype=float)
    return float(np.linalg.norm(forces, axis=1).max()) if forces.size else np.nan


def read_frames(path: Path) -> list[Atoms]:
    frames = aseio.read(str(path), index=":")
    return frames if isinstance(frames, list) else [frames]


def set_aqcat_context(atoms: Atoms) -> Atoms:
    prepared = rebuild_atoms(atoms, slab=True)
    prepared.info["is_spin_off"] = bool(IS_SPIN_OFF)
    prepared.info["is_low_fi"] = bool(IS_LOW_FIDELITY)
    return prepared


def attach_reused_calculator(atoms: Atoms) -> Atoms:
    try:
        AQCAT_CALC.reset()
    except Exception:
        pass
    atoms.calc = AQCAT_CALC
    return atoms


def final_with_results(atoms: Atoms) -> Atoms:
    energy = float(atoms.get_potential_energy())
    forces = np.asarray(atoms.get_forces(), dtype=float)
    final = atoms.copy()
    final.calc = SinglePointCalculator(final, energy=energy, forces=forces)
    return final


def relax_structure(
    atoms: Atoms,
    trajectory_path: Path,
    log_path: Path,
) -> dict:
    trajectory_path.parent.mkdir(parents=True, exist_ok=True)
    log_path.parent.mkdir(parents=True, exist_ok=True)

    base = {
        "status": "failed",
        "error": "",
        "elapsed_s": np.nan,
        "n_frames": 0,
        "aqcat_model_energy_eV": np.nan,
        "final_fmax_eV_A": np.nan,
        "converged": False,
        "trajectory": str(trajectory_path),
        "log": str(log_path),
        "spin_context": SPIN_CONTEXT,
        "is_spin_off": IS_SPIN_OFF,
        "is_low_fi": IS_LOW_FIDELITY,
    }

    if trajectory_path.exists() and not OVERWRITE_EXISTING:
        try:
            frames = read_frames(trajectory_path)
            final = frames[-1]
            energy = float(final.get_potential_energy())
            fmax = maximum_force(final.get_forces())
            return {
                **base,
                "status": "reused",
                "n_frames": len(frames),
                "aqcat_model_energy_eV": energy,
                "final_fmax_eV_A": fmax,
                "converged": bool(np.isfinite(fmax) and fmax < FMAX_THRESHOLD),
            }
        except Exception:
            trajectory_path.unlink(missing_ok=True)

    working = set_aqcat_context(atoms)
    ensure_finite_geometry(working, trajectory_path.stem)
    if has_bad_contact(working):
        raise RuntimeError(
            f"Initial structure contains a distance below "
            f"{MIN_INTERATOMIC_DISTANCE_A:.2f} Å."
        )

    working = attach_reused_calculator(working)
    start = time.time()

    try:
        optimizer = LBFGS(
            working,
            trajectory=str(trajectory_path),
            logfile=str(log_path),
            maxstep=OPTIMIZER_MAXSTEP_A,
        )
        optimizer.run(fmax=FMAX_THRESHOLD, steps=MAX_RELAX_STEPS)

        final = final_with_results(working)
        frames = read_frames(trajectory_path)
        if not frames:
            frames = [final]
        elif np.max(np.abs(frames[-1].positions - final.positions)) > 1e-10:
            frames.append(final)
        else:
            frames[-1] = final
        aseio.write(str(trajectory_path), frames, format="traj")

        energy = float(final.get_potential_energy())
        fmax = maximum_force(final.get_forces())

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        return {
            **base,
            "status": "completed",
            "elapsed_s": time.time() - start,
            "n_frames": len(frames),
            "aqcat_model_energy_eV": energy,
            "final_fmax_eV_A": fmax,
            "converged": bool(np.isfinite(fmax) and fmax < FMAX_THRESHOLD),
        }
    except Exception as exc:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return {
            **base,
            "elapsed_s": time.time() - start,
            "error": f"{type(exc).__name__}: {exc}",
        }


## 9. Relax the bare slab first

The bare slab establishes the geometry used for all subsequent adsorbate
placements. Its AQCat model energy is recorded for traceability only; it is not
part of the adsorption-energy definition.


In [11]:
clean_trajectory_path = TRAJECTORY_DIR / "bare" / "bare_spin_on.traj"
clean_log_path = LOG_DIR / "bare" / "bare_spin_on.log"

clean_relaxation = relax_structure(
    clean_initial,
    clean_trajectory_path,
    clean_log_path,
)

if clean_relaxation["status"] not in {"completed", "reused"}:
    raise RuntimeError(
        "Clean slab relaxation failed: " + clean_relaxation["error"]
    )
if not clean_relaxation["converged"]:
    raise RuntimeError("The clean slab did not reach the force threshold.")

clean_final = aseio.read(clean_relaxation["trajectory"], index=-1)
clean_model_energy_eV = float(clean_relaxation["aqcat_model_energy_eV"])

clean_relaxation_table = pd.DataFrame(
    [
        {
            "system_id": SYSTEM_ID,
            "surface": selected_surface,
            "structure_type": "bare_slab",
            "AQCat_bare_model_energy_eV": clean_model_energy_eV,
            **clean_relaxation,
        }
    ]
)

print(
    "Bare-slab AQCat model energy (diagnostic only): "
    f"{clean_model_energy_eV:.8f} eV"
)
display(clean_relaxation_table)


Bare-slab AQCat model energy (diagnostic only): -0.74505216 eV


,system_id,surface,structure_type,AQCat_bare_model_energy_eV,status,error,elapsed_s,n_frames,aqcat_model_energy_eV,final_fmax_eV_A,converged,trajectory,log,spin_context,is_spin_off,is_low_fi
0,mp-10260_111_term0_aqcat25_spin_on,mp-10260_111,bare_slab,-0.745052,completed,,1.600472,5,-0.745052,0.049914,True,/content/stageII_aqcat25_ev2_results/mp-10260_...,/content/stageII_aqcat25_ev2_results/mp-10260_...,spin_on,False,False


In [12]:
# Reuse the generated FairChem Slab metadata and replace only its atoms.
# This avoids depending on an internal Slab constructor signature.
# All O*, OH*, and OOH* placements therefore start from the same
# AQCat-relaxed bare geometry.

relaxed_slab_object = copy.deepcopy(raw_slab)
relaxed_slab_object.atoms = rebuild_atoms(clean_final, slab=True)

relaxed_slab_path = SYSTEM_DIR / "clean_slab_aqcat_relaxed.traj"
aseio.write(relaxed_slab_path, relaxed_slab_object.atoms)

print("Adsorbate placements will use:", relaxed_slab_path)


Adsorbate placements will use: /content/stageII_aqcat25_ev2_results/mp-10260_111_term0_aqcat25_spin_on/clean_slab_aqcat_relaxed.traj


## 10. Enumerate adsorption configurations

For each ORR intermediate, deterministic heuristic placements are combined with
random-site heuristic placements. Initial configurations with non-finite
geometry or severe atomic overlap are rejected before model inference.


In [13]:
placement_rows = []
configuration_records = []

for adsorbate_name in ADSORBATE_NAMES:
    adsorbate = adsorbates[adsorbate_name]

    heuristic = AdsorbateSlabConfig(
        relaxed_slab_object,
        adsorbate,
        mode="heuristic",
    )
    random_sites = AdsorbateSlabConfig(
        relaxed_slab_object,
        adsorbate,
        mode="random_site_heuristic_placement",
        num_sites=RANDOM_SITES_PER_ADSORBATE,
    )

    groups = [
        ("heuristic", list(heuristic.atoms_list), list(heuristic.metadata_list)),
        (
            "random_site",
            list(random_sites.atoms_list),
            list(random_sites.metadata_list),
        ),
    ]

    running_index = 0
    seen = set()

    for placement_kind, atoms_list, metadata_list in groups:
        if len(atoms_list) != len(metadata_list):
            raise RuntimeError("AdsorbML atoms/metadata length mismatch.")

        accepted_in_group = 0
        for local_index, (atoms, metadata) in enumerate(
            zip(atoms_list, metadata_list)
        ):
            prepared = rebuild_atoms(atoms, slab=True)
            ensure_finite_geometry(prepared, f"{adsorbate_name}_{running_index}")

            geometry_key = (
                tuple(prepared.get_chemical_symbols()),
                tuple(np.round(prepared.positions.ravel(), 2)),
            )
            if geometry_key in seen:
                continue
            seen.add(geometry_key)

            if has_bad_contact(prepared):
                continue

            site = np.asarray(metadata.get("site", [np.nan] * 3), dtype=float)
            configuration_records.append(
                {
                    "config_id": f"{adsorbate_name}_{running_index:03d}",
                    "adsorbate": adsorbate_name,
                    "config_index": running_index,
                    "placement_kind": placement_kind,
                    "placement_local_index": local_index,
                    "initial_site_x_A": float(site[0]),
                    "initial_site_y_A": float(site[1]),
                    "initial_site_z_A": float(site[2]),
                    "sampled_angles": repr(metadata.get("xyz_angles")),
                    "atoms": prepared,
                }
            )
            running_index += 1
            accepted_in_group += 1

        placement_rows.append(
            {
                "adsorbate": adsorbate_name,
                "placement_kind": placement_kind,
                "n_configurations": accepted_in_group,
            }
        )

placement_summary = (
    pd.DataFrame(placement_rows)
    .pivot(
        index="adsorbate",
        columns="placement_kind",
        values="n_configurations",
    )
    .fillna(0)
    .astype(int)
    .reset_index()
)
placement_summary["total"] = (
    placement_summary.get("heuristic", 0)
    + placement_summary.get("random_site", 0)
)

configuration_table = pd.DataFrame(
    [
        {key: value for key, value in record.items() if key != "atoms"}
        for record in configuration_records
    ]
)

display(placement_summary)
display(configuration_table.head())


placement_kind,adsorbate,heuristic,random_site,total
0,O,7,20,27
1,OH,7,20,27
2,OOH,7,20,27


,config_id,adsorbate,config_index,placement_kind,placement_local_index,initial_site_x_A,initial_site_y_A,initial_site_z_A,sampled_angles
0,O_000,O,0,heuristic,0,4.938894,6.110139,19.183581,"array([0, 0, 0])"
1,O_001,O,1,heuristic,1,0.000665,0.000360,20.012293,"array([0, 0, 0])"
2,O_002,O,2,heuristic,2,2.116618,1.221912,19.112187,"array([0, 0, 0])"
3,O_003,O,3,heuristic,3,5.291387,5.498932,18.769225,"array([0, 0, 0])"
4,O_004,O,4,heuristic,4,4.232893,2.444442,18.424905,"array([0, 0, 0])"


## 11. AQCat25-EV2 adsorption-energy relaxation

The model output for each adsorbate-slab structure is already the referenced
AQCat adsorption-energy target. No bare-slab subtraction and no additional CHE
reference calculation are applied.


In [14]:
relaxation_rows = []

for record in tqdm(configuration_records, desc="AQCat25-EV2 relaxations"):
    result = relax_structure(
        record["atoms"],
        (
            TRAJECTORY_DIR
            / record["adsorbate"]
            / f"{record['config_id']}_spin_on.traj"
        ),
        (
            LOG_DIR
            / record["adsorbate"]
            / f"{record['config_id']}_spin_on.log"
        ),
    )
    relaxation_rows.append(
        {
            "system_id": SYSTEM_ID,
            "config_id": record["config_id"],
            "adsorbate": record["adsorbate"],
            "config_index": int(record["config_index"]),
            "placement_kind": record["placement_kind"],
            **result,
        }
    )

relaxation_status = pd.DataFrame(relaxation_rows)
relaxation_status["AQCat_pred_ads_energy_eV"] = pd.to_numeric(
    relaxation_status["aqcat_model_energy_eV"],
    errors="coerce",
)
relaxation_status["energy_interpretation"] = (
    "Direct AQCat25-EV2 adsorption-energy prediction; no bare-slab or "
    "molecular-reference subtraction."
)

energy_columns = [
    "adsorbate",
    "config_id",
    "status",
    "converged",
    "spin_context",
    "AQCat_pred_ads_energy_eV",
    "final_fmax_eV_A",
]

display(relaxation_status[energy_columns])
display(
    relaxation_status.groupby(
        ["adsorbate", "status", "converged"],
        dropna=False,
    )
    .size()
    .rename("count")
    .reset_index()
)


AQCat25-EV2 relaxations:   0%|          | 0/81 [00:00<?, ?it/s]

,adsorbate,config_id,status,converged,spin_context,AQCat_pred_ads_energy_eV,final_fmax_eV_A
0,O,O_000,completed,True,spin_on,1.813784,0.040197
1,O,O_001,completed,True,spin_on,2.232629,0.035218
2,O,O_002,completed,True,spin_on,2.427112,0.033027
3,O,O_003,completed,True,spin_on,2.029887,0.044062
4,O,O_004,completed,True,spin_on,2.667281,0.035624
...,...,...,...,...,...,...,...
76,OOH,OOH_022,completed,True,spin_on,3.821570,0.043103
77,OOH,OOH_023,completed,True,spin_on,3.813610,0.041346
78,OOH,OOH_024,completed,True,spin_on,3.806223,0.040925
79,OOH,OOH_025,completed,True,spin_on,4.096508,0.042627


,adsorbate,status,converged,count
0,O,completed,True,27
1,OH,completed,True,27
2,OOH,completed,True,27


## 12. Trajectory validation

Completed trajectories are rejected when the force threshold is not reached or
when FairChem detects adsorbate dissociation, desorption, intercalation, or a
major surface reconstruction.


In [15]:
def adsorbate_indices_from_tags(
    atoms: Atoms,
    adsorbate_name: str,
) -> np.ndarray:
    tags = np.asarray(atoms.get_tags(), dtype=int)
    tagged = np.flatnonzero(tags == 2)
    expected = ADSORBATE_SIZES[adsorbate_name]
    if len(tagged) == expected:
        return tagged

    fallback = np.arange(len(atoms) - expected, len(atoms), dtype=int)
    tags[fallback] = 2
    atoms.set_tags(tags)
    return fallback


def safe_anomaly_flags(frames: list[Atoms], adsorbate_name: str) -> dict:
    initial = frames[0].copy()
    final = frames[-1].copy()

    if (
        not np.isfinite(initial.positions).all()
        or not np.isfinite(final.positions).all()
        or not np.isfinite(final.cell.array).all()
    ):
        return {
            "adsorbate_dissociated": True,
            "adsorbate_desorbed": True,
            "surface_changed": True,
            "adsorbate_intercalated": True,
            "anomaly_check_error": "non_finite_geometry",
        }

    adsorbate_indices_from_tags(initial, adsorbate_name)
    adsorbate_indices_from_tags(final, adsorbate_name)

    try:
        detector = DetectTrajAnomaly(
            initial,
            final,
            np.asarray(initial.get_tags(), dtype=int),
        )
        return {
            "adsorbate_dissociated": bool(detector.is_adsorbate_dissociated()),
            "adsorbate_desorbed": bool(detector.is_adsorbate_desorbed()),
            "surface_changed": bool(detector.has_surface_changed()),
            "adsorbate_intercalated": bool(detector.is_adsorbate_intercalated()),
            "anomaly_check_error": "",
        }
    except Exception as exc:
        return {
            "adsorbate_dissociated": True,
            "adsorbate_desorbed": True,
            "surface_changed": True,
            "adsorbate_intercalated": True,
            "anomaly_check_error": f"{type(exc).__name__}: {exc}",
        }


def write_poscar(atoms: Atoms, directory: Path) -> Path:
    directory.mkdir(parents=True, exist_ok=True)
    path = directory / "POSCAR"
    aseio.write(
        path,
        atoms,
        format="vasp",
        direct=True,
        vasp5=True,
        sort=True,
    )
    return path


validation_rows = []

for row in relaxation_status.itertuples(index=False):
    record = {
        "system_id": row.system_id,
        "config_id": row.config_id,
        "adsorbate": row.adsorbate,
        "config_index": int(row.config_index),
        "placement_kind": row.placement_kind,
        "trajectory": row.trajectory,
        "relaxation_status": row.status,
        "validation_status": "rejected",
        "rejection_reason": "",
        "aqcat_model_energy_eV": row.aqcat_model_energy_eV,
        "AQCat_pred_ads_energy_eV": row.AQCat_pred_ads_energy_eV,
        "spin_context": row.spin_context,
        "is_spin_off": row.is_spin_off,
        "is_low_fi": row.is_low_fi,
        "final_fmax_eV_A": row.final_fmax_eV_A,
        "n_frames": int(row.n_frames),
        "final_poscar": "",
        "adsorbate_dissociated": False,
        "adsorbate_desorbed": False,
        "surface_changed": False,
        "adsorbate_intercalated": False,
        "anomaly_check_error": "",
    }

    if row.status not in {"completed", "reused"}:
        record["rejection_reason"] = row.error or "relaxation_failed"
        validation_rows.append(record)
        continue

    try:
        frames = read_frames(Path(row.trajectory))
        final = frames[-1]
        record["final_poscar"] = str(
            write_poscar(
                final,
                POSCAR_DIR
                / "all_final_frames"
                / row.adsorbate
                / row.config_id,
            )
        )

        flags = safe_anomaly_flags(frames, row.adsorbate)
        record.update(flags)
        active = [
            key
            for key in (
                "adsorbate_dissociated",
                "adsorbate_desorbed",
                "surface_changed",
                "adsorbate_intercalated",
            )
            if flags[key]
        ]

        if not np.isfinite(row.AQCat_pred_ads_energy_eV):
            record["rejection_reason"] = "non_finite_energy"
        elif not np.isfinite(row.final_fmax_eV_A):
            record["rejection_reason"] = "non_finite_force"
        elif row.final_fmax_eV_A >= FMAX_THRESHOLD:
            record["rejection_reason"] = (
                f"force_above_threshold:{row.final_fmax_eV_A:.6f}"
            )
        elif active:
            record["rejection_reason"] = ";".join(active)
        else:
            record["validation_status"] = "accepted"
    except Exception as exc:
        record["rejection_reason"] = (
            f"validation_failed:{type(exc).__name__}:{exc}"
        )

    validation_rows.append(record)

trajectory_validation = pd.DataFrame(validation_rows)
display(trajectory_validation)
display(
    trajectory_validation.groupby(["adsorbate", "validation_status"])
    .size()
    .rename("count")
    .reset_index()
)


,system_id,config_id,adsorbate,config_index,placement_kind,trajectory,relaxation_status,validation_status,rejection_reason,aqcat_model_energy_eV,...,is_spin_off,is_low_fi,final_fmax_eV_A,n_frames,final_poscar,adsorbate_dissociated,adsorbate_desorbed,surface_changed,adsorbate_intercalated,anomaly_check_error
0,mp-10260_111_term0_aqcat25_spin_on,O_000,O,0,heuristic,/content/stageII_aqcat25_ev2_results/mp-10260_...,completed,accepted,,1.813784,...,False,False,0.040197,32,/content/stageII_aqcat25_ev2_results/mp-10260_...,False,False,False,False,
1,mp-10260_111_term0_aqcat25_spin_on,O_001,O,1,heuristic,/content/stageII_aqcat25_ev2_results/mp-10260_...,completed,accepted,,2.232629,...,False,False,0.035218,12,/content/stageII_aqcat25_ev2_results/mp-10260_...,False,False,False,False,
2,mp-10260_111_term0_aqcat25_spin_on,O_002,O,2,heuristic,/content/stageII_aqcat25_ev2_results/mp-10260_...,completed,accepted,,2.427112,...,False,False,0.033027,11,/content/stageII_aqcat25_ev2_results/mp-10260_...,False,False,False,False,
3,mp-10260_111_term0_aqcat25_spin_on,O_003,O,3,heuristic,/content/stageII_aqcat25_ev2_results/mp-10260_...,completed,accepted,,2.029887,...,False,False,0.044062,23,/content/stageII_aqcat25_ev2_results/mp-10260_...,False,False,False,False,
4,mp-10260_111_term0_aqcat25_spin_on,O_004,O,4,heuristic,/content/stageII_aqcat25_ev2_results/mp-10260_...,completed,accepted,,2.667281,...,False,False,0.035624,3,/content/stageII_aqcat25_ev2_results/mp-10260_...,False,False,False,False,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76,mp-10260_111_term0_aqcat25_spin_on,OOH_022,OOH,22,random_site,/content/stageII_aqcat25_ev2_results/mp-10260_...,completed,accepted,,3.821570,...,False,False,0.043103,58,/content/stageII_aqcat25_ev2_results/mp-10260_...,False,False,False,False,
77,mp-10260_111_term0_aqcat25_spin_on,OOH_023,OOH,23,random_site,/content/stageII_aqcat25_ev2_results/mp-10260_...,completed,accepted,,3.813610,...,False,False,0.041346,74,/content/stageII_aqcat25_ev2_results/mp-10260_...,False,False,False,False,
78,mp-10260_111_term0_aqcat25_spin_on,OOH_024,OOH,24,random_site,/content/stageII_aqcat25_ev2_results/mp-10260_...,completed,accepted,,3.806223,...,False,False,0.040925,69,/content/stageII_aqcat25_ev2_results/mp-10260_...,False,False,False,False,
79,mp-10260_111_term0_aqcat25_spin_on,OOH_025,OOH,25,random_site,/content/stageII_aqcat25_ev2_results/mp-10260_...,completed,accepted,,4.096508,...,False,False,0.042627,70,/content/stageII_aqcat25_ev2_results/mp-10260_...,False,False,False,False,


,adsorbate,validation_status,count
0,O,accepted,27
1,OH,accepted,27
2,OOH,accepted,27


## 13. Adsorption-site and uniqueness analysis

The binding oxygen is assigned to top, bridge, three-fold, or four-fold surface
coordination using periodic in-plane geometry. Configurations are grouped by
specific site instance and by chemical site family before minimum-energy
selection.


In [16]:
LOCAL_XY_RADIUS = 4.8
SURFACE_LAYER_TOL = 1.8
TOP_XY_TOL = 0.45
BRIDGE_PROJECTION_TOL = 0.65
MAX_BRIDGE_LENGTH = 4.5
MAX_HOLLOW_EDGE = 4.8
BOND_BUFFER = 0.65
MAX_BOND_DISTANCE = 3.1
DISTANCE_TIE_TOL = 0.22

def mic_vectors_xy(
    atoms: Atoms,
    anchor_position: np.ndarray,
    positions: np.ndarray,
) -> np.ndarray:
    cell = np.asarray(atoms.cell.array, dtype=float)
    inverse_cell = np.linalg.inv(cell)
    anchor_fractional = anchor_position @ inverse_cell
    fractional = positions @ inverse_cell
    delta = fractional - anchor_fractional
    delta[:, 0] -= np.round(delta[:, 0])
    delta[:, 1] -= np.round(delta[:, 1])
    return delta @ cell

def point_segment_distance_2d(
    point: np.ndarray,
    start: np.ndarray,
    end: np.ndarray,
) -> tuple[float, float]:
    segment = end - start
    denominator = float(np.dot(segment, segment))
    if denominator < 1e-12:
        return float(np.linalg.norm(point - start)), 0.0
    parameter = float(np.dot(point - start, segment) / denominator)
    clipped = float(np.clip(parameter, 0.0, 1.0))
    closest = start + clipped * segment
    return float(np.linalg.norm(point - closest)), clipped

def point_in_triangle_2d(
    point: np.ndarray,
    a: np.ndarray,
    b: np.ndarray,
    c: np.ndarray,
    epsilon: float = 1e-8,
) -> bool:
    v0 = c - a
    v1 = b - a
    v2 = point - a
    denominator = v0[0] * v1[1] - v1[0] * v0[1]
    if abs(denominator) < 1e-12:
        return False
    u = (v2[0] * v1[1] - v1[0] * v2[1]) / denominator
    v = (v0[0] * v2[1] - v2[0] * v0[1]) / denominator
    return bool(u >= -epsilon and v >= -epsilon and u + v <= 1.0 + epsilon)

def site_type_from_coordination(coordination: int) -> str:
    return {
        1: "top",
        2: "bridge",
        3: "3-fold",
        4: "4-fold",
    }.get(coordination, "unknown" if coordination <= 0 else f"{coordination}-fold")

def format_site_result(selected: list[dict], reason: str) -> dict:
    selected = sorted(selected, key=lambda item: item["index"])
    indices = [int(item["index"]) for item in selected]
    symbols = [item["symbol"] for item in selected]
    distances = [float(item["r3d"]) for item in selected]

    site_type = site_type_from_coordination(len(selected))
    neighbor_composition = "-".join(sorted(symbols))
    distance_fingerprint = "-".join(f"{x:.2f}" for x in sorted(distances))
    site_instance_key = f"{site_type}__idx_" + "-".join(map(str, indices))
    site_family = f"{site_type}__{neighbor_composition}"

    return {
        "site_type": site_type,
        "coordination": len(selected),
        "neighbor_composition": neighbor_composition,
        "neighbor_indices": "-".join(map(str, indices)),
        "neighbor_distances_A": ";".join(f"{x:.4f}" for x in distances),
        "site_instance_key": site_instance_key,
        "site_family": site_family,
        "local_site_descriptor": f"{site_family}__d_{distance_fingerprint}",
        "site_assignment_reason": reason,
    }

def choose_binding_oxygen(
    atoms: Atoms,
    adsorbate_indices: np.ndarray,
    surface_indices: np.ndarray,
) -> int:
    symbols = atoms.get_chemical_symbols()
    oxygen_indices = [
        int(i) for i in adsorbate_indices if symbols[i] == "O"
    ]
    if not oxygen_indices:
        raise RuntimeError("No oxygen atom found in the tagged adsorbate.")
    if len(oxygen_indices) == 1:
        return oxygen_indices[0]

    positions = atoms.get_positions()
    surface_positions = positions[surface_indices]
    return min(
        oxygen_indices,
        key=lambda i: float(
            np.linalg.norm(
                mic_vectors_xy(atoms, positions[i], surface_positions),
                axis=1,
            ).min()
        ),
    )

def detect_adsorption_site(atoms: Atoms, adsorbate_name: str) -> dict:
    atoms = atoms.copy()
    adsorbate_indices = adsorbate_indices_from_tags(atoms, adsorbate_name)
    tags = np.asarray(atoms.get_tags(), dtype=int)
    surface_indices = np.flatnonzero(tags == 1)
    if len(surface_indices) == 0:
        surface_indices = np.flatnonzero(tags != 2)

    positions = atoms.get_positions()
    symbols = atoms.get_chemical_symbols()
    binding_oxygen = choose_binding_oxygen(
        atoms,
        adsorbate_indices,
        surface_indices,
    )
    anchor = positions[binding_oxygen]
    surface_positions = positions[surface_indices]
    vectors = mic_vectors_xy(atoms, anchor, surface_positions)
    r3d = np.linalg.norm(vectors, axis=1)
    rxy = np.linalg.norm(vectors[:, :2], axis=1)

    surface_z = surface_positions[:, 2]
    local_mask = (
        (surface_z >= float(surface_z.max()) - SURFACE_LAYER_TOL)
        & (rxy <= LOCAL_XY_RADIUS)
    )
    local_ids = np.flatnonzero(local_mask)
    if len(local_ids) == 0:
        local_ids = np.argsort(r3d)[:12]
    local_ids = sorted(local_ids, key=lambda i: (rxy[i], r3d[i]))[:16]

    candidates = [
        {
            "index": int(surface_indices[i]),
            "symbol": symbols[int(surface_indices[i])],
            "xy": vectors[i, :2],
            "rxy": float(rxy[i]),
            "r3d": float(r3d[i]),
        }
        for i in local_ids
    ]

    origin = np.zeros(2)
    nearest = min(candidates, key=lambda item: item["rxy"])

    if nearest["rxy"] <= TOP_XY_TOL:
        result = format_site_result([nearest], "projection_top")
        result["binding_oxygen_index"] = binding_oxygen
        return result

    bridge_options = []
    for atom_a, atom_b in itertools.combinations(candidates, 2):
        pair_length = float(np.linalg.norm(atom_a["xy"] - atom_b["xy"]))
        if pair_length > MAX_BRIDGE_LENGTH:
            continue
        distance, parameter = point_segment_distance_2d(
            origin,
            atom_a["xy"],
            atom_b["xy"],
        )
        if distance <= BRIDGE_PROJECTION_TOL and 0.0 <= parameter <= 1.0:
            score = (
                distance
                + 0.20 * abs(parameter - 0.5)
                + 0.03 * (atom_a["r3d"] + atom_b["r3d"])
            )
            bridge_options.append((score, atom_a, atom_b))

    if bridge_options:
        _, atom_a, atom_b = min(bridge_options, key=lambda item: item[0])
        result = format_site_result([atom_a, atom_b], "projection_bridge")
        result["binding_oxygen_index"] = binding_oxygen
        return result

    hollow_options = []
    for atom_a, atom_b, atom_c in itertools.combinations(candidates, 3):
        edges = [
            float(np.linalg.norm(atom_a["xy"] - atom_b["xy"])),
            float(np.linalg.norm(atom_a["xy"] - atom_c["xy"])),
            float(np.linalg.norm(atom_b["xy"] - atom_c["xy"])),
        ]
        if max(edges) > MAX_HOLLOW_EDGE:
            continue
        if point_in_triangle_2d(
            origin,
            atom_a["xy"],
            atom_b["xy"],
            atom_c["xy"],
        ):
            centroid = (atom_a["xy"] + atom_b["xy"] + atom_c["xy"]) / 3.0
            score = float(np.linalg.norm(centroid)) + 0.02 * (
                atom_a["r3d"] + atom_b["r3d"] + atom_c["r3d"]
            )
            hollow_options.append((score, atom_a, atom_b, atom_c))

    if hollow_options:
        _, atom_a, atom_b, atom_c = min(
            hollow_options,
            key=lambda item: item[0],
        )
        result = format_site_result(
            [atom_a, atom_b, atom_c],
            "projection_hollow",
        )
        result["binding_oxygen_index"] = binding_oxygen
        return result

    oxygen_radius = covalent_radii[atomic_numbers["O"]]
    bonded = []
    for candidate in candidates:
        surface_radius = covalent_radii[atomic_numbers[candidate["symbol"]]]
        expected = oxygen_radius + surface_radius
        normalized = candidate["r3d"] / expected if expected > 0 else candidate["r3d"]
        if (
            candidate["r3d"] <= expected + BOND_BUFFER
            or candidate["r3d"] <= MAX_BOND_DISTANCE
        ):
            bonded.append((normalized, candidate))

    if bonded:
        minimum_score = min(item[0] for item in bonded)
        selected = [
            item[1]
            for item in bonded
            if item[0] <= minimum_score + DISTANCE_TIE_TOL
        ]
        selected = sorted(selected, key=lambda item: item["r3d"])[:4]
        result = format_site_result(selected, "bond_distance_fallback")
    else:
        result = format_site_result([nearest], "nearest_projection_fallback")

    result["binding_oxygen_index"] = binding_oxygen
    return result


In [17]:
accepted = trajectory_validation[
    trajectory_validation["validation_status"] == "accepted"
].copy()

site_rows = []
for row in accepted.itertuples(index=False):
    final = aseio.read(row.trajectory, index=-1)
    site_rows.append(
        {
            "system_id": row.system_id,
            "config_id": row.config_id,
            "adsorbate": row.adsorbate,
            "config_index": int(row.config_index),
            "placement_kind": row.placement_kind,
            "trajectory": row.trajectory,
            "final_poscar": row.final_poscar,
            "aqcat_model_energy_eV": float(row.aqcat_model_energy_eV),
            "AQCat_pred_ads_energy_eV": float(
                row.AQCat_pred_ads_energy_eV
            ),
            "spin_context": row.spin_context,
            "is_spin_off": bool(row.is_spin_off),
            "is_low_fi": bool(row.is_low_fi),
            "final_fmax_eV_A": float(row.final_fmax_eV_A),
            **detect_adsorption_site(final, row.adsorbate),
        }
    )

all_adsorption_sites = pd.DataFrame(site_rows)

if all_adsorption_sites.empty:
    unique_site_minima = pd.DataFrame()
    site_family_minima = pd.DataFrame()
    global_minima = pd.DataFrame()
else:
    all_adsorption_sites["delta_from_adsorbate_min_eV"] = (
        all_adsorption_sites["AQCat_pred_ads_energy_eV"]
        - all_adsorption_sites.groupby("adsorbate")[
            "AQCat_pred_ads_energy_eV"
        ].transform("min")
    )

    unique_site_minima = (
        all_adsorption_sites
        .sort_values("AQCat_pred_ads_energy_eV")
        .groupby(
            ["system_id", "adsorbate", "site_instance_key"],
            as_index=False,
        )
        .first()
        .sort_values(["adsorbate", "AQCat_pred_ads_energy_eV"])
        .reset_index(drop=True)
    )

    site_family_minima = (
        all_adsorption_sites
        .sort_values("AQCat_pred_ads_energy_eV")
        .groupby(
            ["system_id", "adsorbate", "site_family"],
            as_index=False,
        )
        .first()
        .sort_values(["adsorbate", "AQCat_pred_ads_energy_eV"])
        .reset_index(drop=True)
    )

    global_minima = (
        all_adsorption_sites
        .sort_values("AQCat_pred_ads_energy_eV")
        .groupby(["system_id", "adsorbate"], as_index=False)
        .first()
        .sort_values("adsorbate")
        .reset_index(drop=True)
    )

print("All accepted configurations")
display(all_adsorption_sites)
print("Unique site-instance minima")
display(unique_site_minima)
print("Site-family minima")
display(site_family_minima)
print("Global minimum per adsorbate")
display(global_minima)


All accepted configurations


,system_id,config_id,adsorbate,config_index,placement_kind,trajectory,final_poscar,aqcat_model_energy_eV,AQCat_pred_ads_energy_eV,spin_context,...,coordination,neighbor_composition,neighbor_indices,neighbor_distances_A,site_instance_key,site_family,local_site_descriptor,site_assignment_reason,binding_oxygen_index,delta_from_adsorbate_min_eV
0,mp-10260_111_term0_aqcat25_spin_on,O_000,O,0,heuristic,/content/stageII_aqcat25_ev2_results/mp-10260_...,/content/stageII_aqcat25_ev2_results/mp-10260_...,1.813784,1.813784,spin_on,...,2,Ni-Sb,9-16,1.9898;1.8951,bridge__idx_9-16,bridge__Ni-Sb,bridge__Ni-Sb__d_1.90-1.99,projection_bridge,48,0.046923
1,mp-10260_111_term0_aqcat25_spin_on,O_001,O,1,heuristic,/content/stageII_aqcat25_ev2_results/mp-10260_...,/content/stageII_aqcat25_ev2_results/mp-10260_...,2.232629,2.232629,spin_on,...,1,Sb,9,1.8741,top__idx_9,top__Sb,top__Sb__d_1.87,projection_top,48,0.465768
2,mp-10260_111_term0_aqcat25_spin_on,O_002,O,2,heuristic,/content/stageII_aqcat25_ev2_results/mp-10260_...,/content/stageII_aqcat25_ev2_results/mp-10260_...,2.427112,2.427112,spin_on,...,1,Ni,4,1.8832,top__idx_4,top__Ni,top__Ni__d_1.88,projection_top,48,0.660251
3,mp-10260_111_term0_aqcat25_spin_on,O_003,O,3,heuristic,/content/stageII_aqcat25_ev2_results/mp-10260_...,/content/stageII_aqcat25_ev2_results/mp-10260_...,2.029887,2.029887,spin_on,...,2,Ni-Ni,14-16,2.9539;1.9457,bridge__idx_14-16,bridge__Ni-Ni,bridge__Ni-Ni__d_1.95-2.95,projection_bridge,48,0.263026
4,mp-10260_111_term0_aqcat25_spin_on,O_004,O,4,heuristic,/content/stageII_aqcat25_ev2_results/mp-10260_...,/content/stageII_aqcat25_ev2_results/mp-10260_...,2.667281,2.667281,spin_on,...,1,Ni,2,1.9593,top__idx_2,top__Ni,top__Ni__d_1.96,projection_top,48,0.900420
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76,mp-10260_111_term0_aqcat25_spin_on,OOH_022,OOH,22,random_site,/content/stageII_aqcat25_ev2_results/mp-10260_...,/content/stageII_aqcat25_ev2_results/mp-10260_...,3.821570,3.821570,spin_on,...,2,Sb-Sb,21-45,4.1367;2.1318,bridge__idx_21-45,bridge__Sb-Sb,bridge__Sb-Sb__d_2.13-4.14,projection_bridge,48,0.033748
77,mp-10260_111_term0_aqcat25_spin_on,OOH_023,OOH,23,random_site,/content/stageII_aqcat25_ev2_results/mp-10260_...,/content/stageII_aqcat25_ev2_results/mp-10260_...,3.813610,3.813610,spin_on,...,1,Sb,21,2.1327,top__idx_21,top__Sb,top__Sb__d_2.13,projection_top,48,0.025787
78,mp-10260_111_term0_aqcat25_spin_on,OOH_024,OOH,24,random_site,/content/stageII_aqcat25_ev2_results/mp-10260_...,/content/stageII_aqcat25_ev2_results/mp-10260_...,3.806223,3.806223,spin_on,...,1,Sb,9,2.1312,top__idx_9,top__Sb,top__Sb__d_2.13,projection_top,48,0.018401
79,mp-10260_111_term0_aqcat25_spin_on,OOH_025,OOH,25,random_site,/content/stageII_aqcat25_ev2_results/mp-10260_...,/content/stageII_aqcat25_ev2_results/mp-10260_...,4.096508,4.096508,spin_on,...,2,Ni-Ni,4-16,4.0386;2.1164,bridge__idx_4-16,bridge__Ni-Ni,bridge__Ni-Ni__d_2.12-4.04,projection_bridge,48,0.308685


Unique site-instance minima


,system_id,adsorbate,site_instance_key,config_id,config_index,placement_kind,trajectory,final_poscar,aqcat_model_energy_eV,AQCat_pred_ads_energy_eV,...,site_type,coordination,neighbor_composition,neighbor_indices,neighbor_distances_A,site_family,local_site_descriptor,site_assignment_reason,binding_oxygen_index,delta_from_adsorbate_min_eV
0,mp-10260_111_term0_aqcat25_spin_on,O,bridge__idx_4-26,O_009,9,random_site,/content/stageII_aqcat25_ev2_results/mp-10260_...,/content/stageII_aqcat25_ev2_results/mp-10260_...,1.766861,1.766861,...,bridge,2,Ni-Ni,4-26,1.8777;1.9582,bridge__Ni-Ni,bridge__Ni-Ni__d_1.88-1.96,projection_bridge,48,0.000000
1,mp-10260_111_term0_aqcat25_spin_on,O,bridge__idx_9-16,O_000,0,heuristic,/content/stageII_aqcat25_ev2_results/mp-10260_...,/content/stageII_aqcat25_ev2_results/mp-10260_...,1.813784,1.813784,...,bridge,2,Ni-Sb,9-16,1.9898;1.8951,bridge__Ni-Sb,bridge__Ni-Sb__d_1.90-1.99,projection_bridge,48,0.046923
2,mp-10260_111_term0_aqcat25_spin_on,O,bridge__idx_16-21,O_007,7,random_site,/content/stageII_aqcat25_ev2_results/mp-10260_...,/content/stageII_aqcat25_ev2_results/mp-10260_...,1.814762,1.814762,...,bridge,2,Ni-Sb,16-21,1.8971;1.9891,bridge__Ni-Sb,bridge__Ni-Sb__d_1.90-1.99,projection_bridge,48,0.047901
3,mp-10260_111_term0_aqcat25_spin_on,O,bridge__idx_40-45,O_013,13,random_site,/content/stageII_aqcat25_ev2_results/mp-10260_...,/content/stageII_aqcat25_ev2_results/mp-10260_...,1.817736,1.817736,...,bridge,2,Ni-Sb,40-45,1.8921;1.9907,bridge__Ni-Sb,bridge__Ni-Sb__d_1.89-1.99,projection_bridge,48,0.050875
4,mp-10260_111_term0_aqcat25_spin_on,O,bridge__idx_16-45,O_014,14,random_site,/content/stageII_aqcat25_ev2_results/mp-10260_...,/content/stageII_aqcat25_ev2_results/mp-10260_...,1.819860,1.819860,...,bridge,2,Ni-Sb,16-45,1.8904;1.9947,bridge__Ni-Sb,bridge__Ni-Sb__d_1.89-1.99,projection_bridge,48,0.052999
5,mp-10260_111_term0_aqcat25_spin_on,O,bridge__idx_4-21,O_020,20,random_site,/content/stageII_aqcat25_ev2_results/mp-10260_...,/content/stageII_aqcat25_ev2_results/mp-10260_...,1.820936,1.820936,...,bridge,2,Ni-Sb,4-21,1.8928;1.9936,bridge__Ni-Sb,bridge__Ni-Sb__d_1.89-1.99,projection_bridge,48,0.054075
6,mp-10260_111_term0_aqcat25_spin_on,O,bridge__idx_33-40,O_025,25,random_site,/content/stageII_aqcat25_ev2_results/mp-10260_...,/content/stageII_aqcat25_ev2_results/mp-10260_...,1.821587,1.821587,...,bridge,2,Ni-Sb,33-40,1.9908;1.8986,bridge__Ni-Sb,bridge__Ni-Sb__d_1.90-1.99,projection_bridge,48,0.054726
7,mp-10260_111_term0_aqcat25_spin_on,O,bridge__idx_21-40,O_010,10,random_site,/content/stageII_aqcat25_ev2_results/mp-10260_...,/content/stageII_aqcat25_ev2_results/mp-10260_...,1.822167,1.822167,...,bridge,2,Ni-Sb,21-40,1.9951;1.8918,bridge__Ni-Sb,bridge__Ni-Sb__d_1.89-2.00,projection_bridge,48,0.055306
8,mp-10260_111_term0_aqcat25_spin_on,O,bridge__idx_4-9,O_022,22,random_site,/content/stageII_aqcat25_ev2_results/mp-10260_...,/content/stageII_aqcat25_ev2_results/mp-10260_...,1.822661,1.822661,...,bridge,2,Ni-Sb,4-9,1.8861;1.9918,bridge__Ni-Sb,bridge__Ni-Sb__d_1.89-1.99,projection_bridge,48,0.055800
9,mp-10260_111_term0_aqcat25_spin_on,O,bridge__idx_28-33,O_021,21,random_site,/content/stageII_aqcat25_ev2_results/mp-10260_...,/content/stageII_aqcat25_ev2_results/mp-10260_...,1.824833,1.824833,...,bridge,2,Ni-Sb,28-33,1.8951;1.9901,bridge__Ni-Sb,bridge__Ni-Sb__d_1.90-1.99,projection_bridge,48,0.057972


Site-family minima


,system_id,adsorbate,site_family,config_id,config_index,placement_kind,trajectory,final_poscar,aqcat_model_energy_eV,AQCat_pred_ads_energy_eV,...,site_type,coordination,neighbor_composition,neighbor_indices,neighbor_distances_A,site_instance_key,local_site_descriptor,site_assignment_reason,binding_oxygen_index,delta_from_adsorbate_min_eV
0,mp-10260_111_term0_aqcat25_spin_on,O,bridge__Ni-Ni,O_009,9,random_site,/content/stageII_aqcat25_ev2_results/mp-10260_...,/content/stageII_aqcat25_ev2_results/mp-10260_...,1.766861,1.766861,...,bridge,2,Ni-Ni,4-26,1.8777;1.9582,bridge__idx_4-26,bridge__Ni-Ni__d_1.88-1.96,projection_bridge,48,0.000000
1,mp-10260_111_term0_aqcat25_spin_on,O,bridge__Ni-Sb,O_000,0,heuristic,/content/stageII_aqcat25_ev2_results/mp-10260_...,/content/stageII_aqcat25_ev2_results/mp-10260_...,1.813784,1.813784,...,bridge,2,Ni-Sb,9-16,1.9898;1.8951,bridge__idx_9-16,bridge__Ni-Sb__d_1.90-1.99,projection_bridge,48,0.046923
2,mp-10260_111_term0_aqcat25_spin_on,O,top__Sb,O_006,6,heuristic,/content/stageII_aqcat25_ev2_results/mp-10260_...,/content/stageII_aqcat25_ev2_results/mp-10260_...,2.219213,2.219213,...,top,1,Sb,9,1.8715,top__idx_9,top__Sb__d_1.87,projection_top,48,0.452352
3,mp-10260_111_term0_aqcat25_spin_on,O,top__Ni,O_002,2,heuristic,/content/stageII_aqcat25_ev2_results/mp-10260_...,/content/stageII_aqcat25_ev2_results/mp-10260_...,2.427112,2.427112,...,top,1,Ni,4,1.8832,top__idx_4,top__Ni__d_1.88,projection_top,48,0.660251
4,mp-10260_111_term0_aqcat25_spin_on,OH,top__Sb,OH_025,25,random_site,/content/stageII_aqcat25_ev2_results/mp-10260_...,/content/stageII_aqcat25_ev2_results/mp-10260_...,0.835186,0.835186,...,top,1,Sb,33,2.0411,top__idx_33,top__Sb__d_2.04,projection_top,48,0.000000
5,mp-10260_111_term0_aqcat25_spin_on,OH,bridge__Ni-Sb,OH_010,10,random_site,/content/stageII_aqcat25_ev2_results/mp-10260_...,/content/stageII_aqcat25_ev2_results/mp-10260_...,1.122219,1.122219,...,bridge,2,Ni-Sb,28-33,2.1586;2.1831,bridge__idx_28-33,bridge__Ni-Sb__d_2.16-2.18,projection_bridge,48,0.287033
6,mp-10260_111_term0_aqcat25_spin_on,OOH,top__Sb,OOH_007,7,random_site,/content/stageII_aqcat25_ev2_results/mp-10260_...,/content/stageII_aqcat25_ev2_results/mp-10260_...,3.787823,3.787823,...,top,1,Sb,21,2.1207,top__idx_21,top__Sb__d_2.12,projection_top,48,0.000000
7,mp-10260_111_term0_aqcat25_spin_on,OOH,bridge__Sb-Sb,OOH_003,3,heuristic,/content/stageII_aqcat25_ev2_results/mp-10260_...,/content/stageII_aqcat25_ev2_results/mp-10260_...,3.804814,3.804814,...,bridge,2,Sb-Sb,9-21,2.1266;4.2357,bridge__idx_9-21,bridge__Sb-Sb__d_2.13-4.24,projection_bridge,48,0.016991
8,mp-10260_111_term0_aqcat25_spin_on,OOH,bridge__Ni-Sb,OOH_015,15,random_site,/content/stageII_aqcat25_ev2_results/mp-10260_...,/content/stageII_aqcat25_ev2_results/mp-10260_...,3.841142,3.841142,...,bridge,2,Ni-Sb,40-45,3.3556;2.1281,bridge__idx_40-45,bridge__Ni-Sb__d_2.13-3.36,projection_bridge,48,0.053319
9,mp-10260_111_term0_aqcat25_spin_on,OOH,bridge__Ni-Ni,OOH_017,17,random_site,/content/stageII_aqcat25_ev2_results/mp-10260_...,/content/stageII_aqcat25_ev2_results/mp-10260_...,4.079521,4.079521,...,bridge,2,Ni-Ni,4-28,3.9799;2.1446,bridge__idx_4-28,bridge__Ni-Ni__d_2.14-3.98,projection_bridge,48,0.291698


Global minimum per adsorbate


,system_id,adsorbate,config_id,config_index,placement_kind,trajectory,final_poscar,aqcat_model_energy_eV,AQCat_pred_ads_energy_eV,spin_context,...,coordination,neighbor_composition,neighbor_indices,neighbor_distances_A,site_instance_key,site_family,local_site_descriptor,site_assignment_reason,binding_oxygen_index,delta_from_adsorbate_min_eV
0,mp-10260_111_term0_aqcat25_spin_on,O,O_009,9,random_site,/content/stageII_aqcat25_ev2_results/mp-10260_...,/content/stageII_aqcat25_ev2_results/mp-10260_...,1.766861,1.766861,spin_on,...,2,Ni-Ni,4-26,1.8777;1.9582,bridge__idx_4-26,bridge__Ni-Ni,bridge__Ni-Ni__d_1.88-1.96,projection_bridge,48,0.0
1,mp-10260_111_term0_aqcat25_spin_on,OH,OH_025,25,random_site,/content/stageII_aqcat25_ev2_results/mp-10260_...,/content/stageII_aqcat25_ev2_results/mp-10260_...,0.835186,0.835186,spin_on,...,1,Sb,33,2.0411,top__idx_33,top__Sb,top__Sb__d_2.04,projection_top,48,0.0
2,mp-10260_111_term0_aqcat25_spin_on,OOH,OOH_007,7,random_site,/content/stageII_aqcat25_ev2_results/mp-10260_...,/content/stageII_aqcat25_ev2_results/mp-10260_...,3.787823,3.787823,spin_on,...,1,Sb,21,2.1207,top__idx_21,top__Sb,top__Sb__d_2.12,projection_top,48,0.0


## 14. Verify and summarize AQCat adsorption energies

This consistency check confirms that the published result column is exactly the
direct model output for every accepted adsorbate-slab trajectory.


In [18]:
def verify_direct_aqcat_energy(
    table: pd.DataFrame,
    *,
    table_name: str,
    atol: float = 1e-10,
) -> pd.DataFrame:
    if table.empty:
        return table.copy()

    checked = table.copy()
    model_energy = checked["aqcat_model_energy_eV"].astype(float)
    reported = checked["AQCat_pred_ads_energy_eV"].astype(float)
    finite = np.isfinite(model_energy) & np.isfinite(reported)

    if finite.any() and not np.allclose(
        model_energy.loc[finite],
        reported.loc[finite],
        rtol=0.0,
        atol=atol,
    ):
        maximum_error = float(
            np.max(
                np.abs(
                    model_energy.loc[finite] - reported.loc[finite]
                )
            )
        )
        raise RuntimeError(
            f"Direct AQCat energy check failed for {table_name}; "
            f"maximum mismatch = {maximum_error:.3e} eV."
        )

    checked["direct_energy_check_eV"] = reported - model_energy
    return checked


all_adsorption_sites = verify_direct_aqcat_energy(
    all_adsorption_sites,
    table_name="all_adsorption_sites",
)
unique_site_minima = verify_direct_aqcat_energy(
    unique_site_minima,
    table_name="unique_site_minima",
)
site_family_minima = verify_direct_aqcat_energy(
    site_family_minima,
    table_name="site_family_minima",
)
global_minima = verify_direct_aqcat_energy(
    global_minima,
    table_name="global_minima",
)

required_adsorbates = set(ADSORBATE_NAMES)
available_adsorbates = set(
    global_minima.get("adsorbate", pd.Series(dtype=str)).tolist()
)
missing_adsorbates = required_adsorbates - available_adsorbates

if missing_adsorbates:
    raise RuntimeError(
        "Accepted global minima are required for O, OH, and OOH. Missing: "
        + ", ".join(sorted(missing_adsorbates))
    )

global_adsorption_energies = (
    global_minima
    .set_index("adsorbate")
    .loc[list(ADSORBATE_NAMES)]
    .reset_index()
)

energy_summary_columns = [
    "adsorbate",
    "config_id",
    "site_family",
    "spin_context",
    "AQCat_pred_ads_energy_eV",
    "direct_energy_check_eV",
]

print("Global AQCat25-EV2 adsorption-energy predictions")
display(global_adsorption_energies[energy_summary_columns])


Global AQCat25-EV2 adsorption-energy predictions


,adsorbate,config_id,site_family,spin_context,AQCat_pred_ads_energy_eV,direct_energy_check_eV
0,O,O_009,bridge__Ni-Ni,spin_on,1.766861,0.0
1,OH,OH_025,top__Sb,spin_on,0.835186,0.0
2,OOH,OOH_007,top__Sb,spin_on,3.787823,0.0


## 15. Interactive structural inspection

The viewer displays a 2 x 2 x 1 visual supercell so adsorbates close to periodic
boundaries remain visible. The dropdown label reports the AQCat adsorption
energy, not the bare-slab diagnostic energy.


In [19]:
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass


def atoms_to_cif_string(atoms: Atoms) -> str:
    buffer = io.BytesIO()
    aseio.write(buffer, atoms, format="cif")
    return buffer.getvalue().decode("utf-8")


def show_atoms_py3dmol(
    atoms: Atoms,
    *,
    width: int = 850,
    height: int = 520,
) -> None:
    display_atoms = atoms.repeat((2, 2, 1))

    viewer = py3Dmol.view(width=width, height=height)
    viewer.addModel(atoms_to_cif_string(display_atoms), "cif")
    viewer.setStyle(
        {
            "sphere": {"scale": 0.32},
            "stick": {"radius": 0.14},
        }
    )
    viewer.addUnitCell()
    viewer.zoomTo()
    viewer.show()


structure_catalog = {
    "Clean AQCat-relaxed slab": str(clean_trajectory_path),
}

if not global_minima.empty:
    for row in global_minima.itertuples(index=False):
        label = (
            f"GLOBAL MIN | {row.adsorbate}* | {row.site_family} | "
            f"Eads = {row.AQCat_pred_ads_energy_eV:.4f} eV | "
            f"{row.config_id}"
        )
        structure_catalog[label] = str(row.trajectory)

if not unique_site_minima.empty:
    for row in unique_site_minima.itertuples(index=False):
        label = (
            f"UNIQUE SITE | {row.adsorbate}* | "
            f"{row.site_instance_key} | "
            f"Eads = {row.AQCat_pred_ads_energy_eV:.4f} eV | "
            f"{row.config_id}"
        )
        structure_catalog[label] = str(row.trajectory)

selector = widgets.Dropdown(
    options=list(structure_catalog.keys()),
    value="Clean AQCat-relaxed slab",
    description="Structure:",
    layout=widgets.Layout(width="95%"),
)
viewer_output = widgets.Output()


def render_selected_structure(change=None) -> None:
    label = selector.value
    trajectory_path = Path(structure_catalog[label])

    with viewer_output:
        clear_output(wait=True)
        if not trajectory_path.exists():
            print("Structure file not found:", trajectory_path)
            return

        try:
            atoms = aseio.read(str(trajectory_path), index=-1)
            print(label)
            print("Visualization cell: 2 x 2 x 1")
            show_atoms_py3dmol(atoms)
        except Exception as exc:
            print(f"Could not display {label}")
            print(f"{type(exc).__name__}: {exc}")


selector.observe(render_selected_structure, names="value")
display(widgets.VBox([selector, viewer_output]))
render_selected_structure()


## 16. Repository-ready AQCat RPBE single-point inputs

The exported VASP calculations use the AQCat25 slab electronic-structure
protocol, converted from ionic relaxation to a static single-point calculation.
The retained settings are RPBE, `ENCUT = 500`, `PREC = Accurate`, Gaussian
smearing with `ISMEAR = 0` and `SIGMA = 0.1`, `ALGO = Normal`, `NELM = 250`,
`EDIFF = 1E-4`, `ISYM = 0`, `SYMPREC = 1E-10`, and `LREAL = Auto`.

For the static calculation, ionic-relaxation controls are removed. The INCAR
therefore uses `IBRION = -1` and `NSW = 0` and does not write `ISIF`, `POTIM`,
or `EDIFFG`. No empirical dispersion correction or dipole-correction block is
added because neither belongs to the AQCat25 protocol used here.

Spin polarization is enabled only when the slab contains at least one AQCat25
magnetic element. The initial magnetic moments follow AQCat25 Table 6:

- `5.00 μB`: V, Cr, Mn, Fe, Co, Ni, Mo, W, and Ce
- `2.20 μB`: Ru and Os
- `1.73 μB`: Cu
- `0.00 μB`: all other slab atoms and all adsorbate O/H atoms

The POSCAR is reordered deterministically as slab species first, followed by
adsorbate O and H when present. The `MAGMOM` line is then generated in exactly
that POSCAR atom order and compressed using VASP repetition notation, for
example `36*5.00 15*0.00`.


In [20]:
POTCAR_LABELS = {
    "H": "H",
    "O": "O",
    "Ti": "Ti",
    "V": "V",
    "Cr": "Cr",
    "Mn": "Mn",
    "Fe": "Fe",
    "Co": "Co",
    "Ni": "Ni",
    "Cu": "Cu",
    "Nb": "Nb",
    "Mo": "Mo",
    "Ru": "Ru",
    "Sn": "Sn",
    "Sb": "Sb",
    "Ta": "Ta",
    "W": "W",
    "Os": "Os",
    "Ir": "Ir",
    "Pt": "Pt",
    "Ce": "Ce",
}


def unique_in_order(values: list[str]) -> list[str]:
    """Return unique strings while preserving their first occurrence."""
    output: list[str] = []
    for value in values:
        if value not in output:
            output.append(value)
    return output


def write_poscar_with_selective_dynamics(
    poscar_path: Path,
    atoms: Atoms,
    system_label: str,
    n_adsorbate_atoms: int = 0,
) -> dict:
    """
    Write a deterministic VASP5 POSCAR.

    Atom order is explicit and reproducible:
    1. slab atoms grouped by slab element order;
    2. adsorbate atoms grouped as O then H when present.

    The FairChem O*, OH*, and OOH* structures append the adsorbate atoms to the
    slab, so the final ``n_adsorbate_atoms`` indices are treated as adsorbate.
    """
    n_atoms = len(atoms)
    if not 0 <= n_adsorbate_atoms <= n_atoms:
        raise ValueError(
            "n_adsorbate_atoms must be between 0 and the total atom count."
        )

    symbols = atoms.get_chemical_symbols()
    split_index = n_atoms - n_adsorbate_atoms
    slab_indices = list(range(split_index))
    adsorbate_indices = list(range(split_index, n_atoms))

    slab_symbols = [symbols[index] for index in slab_indices]
    adsorbate_symbols = [symbols[index] for index in adsorbate_indices]

    unexpected_adsorbate = sorted(set(adsorbate_symbols) - {"O", "H"})
    if unexpected_adsorbate:
        raise ValueError(
            "Expected adsorbate atoms to contain only O/H, found: "
            + ", ".join(unexpected_adsorbate)
        )

    slab_element_order = unique_in_order(slab_symbols)
    adsorbate_element_order = [
        element for element in ("O", "H") if element in adsorbate_symbols
    ]

    duplicate_domains = set(slab_element_order) & set(adsorbate_element_order)
    if duplicate_domains:
        raise ValueError(
            "This alloy-slab exporter assumes O/H occur only in the adsorbate. "
            "Duplicate slab/adsorbate elements: "
            + ", ".join(sorted(duplicate_domains))
        )

    element_order = slab_element_order + adsorbate_element_order
    grouped_indices = [
        index
        for element in slab_element_order
        for index in slab_indices
        if symbols[index] == element
    ] + [
        index
        for element in adsorbate_element_order
        for index in adsorbate_indices
        if symbols[index] == element
    ]

    ordered_symbols = [symbols[index] for index in grouped_indices]
    ordered_roles = [
        "adsorbate" if index in set(adsorbate_indices) else "slab"
        for index in grouped_indices
    ]
    counts = [ordered_symbols.count(element) for element in element_order]

    fixed_indices: set[int] = set()
    for constraint in getattr(atoms, "constraints", []):
        if isinstance(constraint, FixAtoms):
            fixed_indices.update(map(int, constraint.get_indices()))

    scaled_positions = atoms.get_scaled_positions(wrap=False)
    cell = np.asarray(atoms.cell.array, dtype=float)

    lines = [
        f"{system_label} | final AQCat25-EV2 spin-on frame",
        "1.0",
    ]
    lines.extend(
        "  " + " ".join(f"{component:.12f}" for component in vector)
        for vector in cell
    )
    lines.append("  " + " ".join(element_order))
    lines.append("  " + " ".join(map(str, counts)))
    lines.append("Selective dynamics")
    lines.append("Direct")

    for index in grouped_indices:
        position = scaled_positions[index]
        flags = "F F F" if index in fixed_indices else "T T T"
        lines.append(
            "  "
            + " ".join(f"{component:.12f}" for component in position)
            + f"  {flags}"
        )

    poscar_path.write_text("\n".join(lines) + "\n", encoding="utf-8")

    return {
        "element_order": element_order,
        "counts": counts,
        "grouped_indices": grouped_indices,
        "ordered_symbols": ordered_symbols,
        "ordered_roles": ordered_roles,
        "n_adsorbate_atoms": n_adsorbate_atoms,
    }


def write_kpoints(directory: Path, atoms: Atoms) -> dict:
    """Write the AQCat reciprocal-length Monkhorst-Pack mesh."""
    cell = np.asarray(atoms.cell.array, dtype=float)
    a_length = float(np.linalg.norm(cell[0]))
    b_length = float(np.linalg.norm(cell[1]))

    if not np.isfinite(a_length) or not np.isfinite(b_length):
        raise ValueError("Non-finite in-plane lattice-vector length.")
    if a_length <= 0.0 or b_length <= 0.0:
        raise ValueError("Invalid in-plane lattice vectors for KPOINTS.")

    mesh = [
        max(1, int(round(KPOINT_DENSITY / a_length))),
        max(1, int(round(KPOINT_DENSITY / b_length))),
        1,
    ]

    path = directory / "KPOINTS"
    path.write_text(
        "Automatic mesh\n"
        "0\n"
        "Monkhorst-Pack\n"
        + " ".join(map(str, mesh))
        + "\n0 0 0\n",
        encoding="utf-8",
    )

    return {
        "path": str(path),
        "mesh": mesh,
        "a_length_A": a_length,
        "b_length_A": b_length,
        "density_parameter": KPOINT_DENSITY,
        "rule": "round(40/|a|), round(40/|b|), 1",
        "scheme": "Monkhorst-Pack",
    }


def aqcat_magnetic_moments(
    ordered_symbols: list[str],
    ordered_roles: list[str],
) -> list[float]:
    """Return AQCat Table 6 moments in exact POSCAR atom order."""
    if len(ordered_symbols) != len(ordered_roles):
        raise ValueError("ordered_symbols and ordered_roles must have equal length.")

    moments: list[float] = []
    for symbol, role in zip(ordered_symbols, ordered_roles):
        if role == "adsorbate":
            moments.append(0.0)
        elif role == "slab":
            moments.append(float(AQCAT_INITIAL_MAGMOM.get(symbol, 0.0)))
        else:
            raise ValueError(f"Unknown atom role: {role}")
    return moments


def spin_is_required(
    ordered_symbols: list[str],
    ordered_roles: list[str],
) -> bool:
    """Enable spin only when an AQCat magnetic element occurs in the slab."""
    return any(
        role == "slab" and symbol in AQCAT_SPIN_ELEMENTS
        for symbol, role in zip(ordered_symbols, ordered_roles)
    )


def compress_magmom(moments: list[float]) -> str:
    """Compress consecutive moments using VASP ``count*value`` notation."""
    if not moments:
        return ""

    groups: list[tuple[int, float]] = []
    count = 1
    current = moments[0]

    for value in moments[1:]:
        if np.isclose(value, current, atol=1e-12, rtol=0.0):
            count += 1
        else:
            groups.append((count, current))
            count = 1
            current = value
    groups.append((count, current))

    return " ".join(
        f"{count}*{value:.2f}" if count > 1 else f"{value:.2f}"
        for count, value in groups
    )


def write_incar(
    directory: Path,
    system_label: str,
    ordered_symbols: list[str],
    ordered_roles: list[str],
) -> tuple[Path, dict]:
    """Write an AQCat-protocol RPBE spin-aware static INCAR."""
    use_spin = spin_is_required(ordered_symbols, ordered_roles)
    ispin = 2 if use_spin else 1
    magmoms = (
        aqcat_magnetic_moments(ordered_symbols, ordered_roles)
        if use_spin
        else []
    )
    magmom_expression = compress_magmom(magmoms) if use_spin else ""

    spin_block = ""
    if use_spin:
        spin_block = f"""
# MAGMOM follows the exact POSCAR atom order.
# Slab atoms use AQCat25 Table 6; adsorbate O/H use 0.00 μB.
MAGMOM = {magmom_expression}
"""

    incar_text = f"""
SYSTEM = {system_label}

# AQCat25 slab electronic-structure protocol
GGA = RP
PREC = {VASP_PREC}
ENCUT = {VASP_ENCUT_EV}
EDIFF = {VASP_EDIFF:.0E}
NELM = {VASP_NELM}
ALGO = Normal
ISMEAR = 0
SIGMA = {VASP_SIGMA_EV}
ISYM = 0
SYMPREC = {VASP_SYMPREC:.0E}
LREAL = Auto

# Spin treatment
ISPIN = {ispin}
{spin_block}
# Static single-point calculation from the AQCat-relaxed geometry
IBRION = -1
NSW = 0

# Output
LORBIT = 11
LWAVE = .TRUE.
LCHARG = .FALSE.
ISTART = {VASP_ISTART}

# AQCat production parallelization convention
NPAR = {VASP_NPAR}
"""

    path = directory / "INCAR"
    path.write_text(
        textwrap.dedent(incar_text).strip() + "\n",
        encoding="utf-8",
    )

    magnetic_elements_present = sorted(
        {
            symbol
            for symbol, role in zip(ordered_symbols, ordered_roles)
            if role == "slab" and symbol in AQCAT_SPIN_ELEMENTS
        }
    )

    return path, {
        "spin_polarized": use_spin,
        "ISPIN": ispin,
        "MAGMOM_per_atom_uB": magmoms,
        "MAGMOM_expression": magmom_expression,
        "magnetic_elements_present": magnetic_elements_present,
        "magmom_source": "AQCat25 Table 6",
        "adsorbate_magmom_uB": 0.0,
    }


def safe_job_name(label: str) -> str:
    """Convert a scientific label to a valid Slurm job name."""
    cleaned = re.sub(r"[^A-Za-z0-9_-]+", "-", label).strip("-")
    return cleaned[:64] or "vasp_aqcat_sp"


def write_job_script(directory: Path, job_label: str) -> Path:
    """Write an editable Slurm template for one static VASP calculation."""
    job_name = safe_job_name(job_label)

    job_text = f"""#!/bin/bash
#SBATCH -N 1
#SBATCH -n 64
#SBATCH --time=24:00:00
#SBATCH -p rome
#SBATCH -J {job_name}
#SBATCH --output=out.%j
#SBATCH --error=err.%j

set -euo pipefail

if grep -q "POTCAR NOT DISTRIBUTED" POTCAR; then
    echo "ERROR: Replace the POTCAR placeholder with a licensed POTCAR."
    exit 2
fi

module purge
module load 2024
module load VASP6/6.4.3-foss-2024a

srun vasp_std > vasp.out
"""

    path = directory / "job.sh"
    path.write_text(textwrap.dedent(job_text), encoding="utf-8")
    path.chmod(0o755)
    return path


def write_potcar_placeholder(
    directory: Path,
    element_order: list[str],
) -> tuple[Path, list[str]]:
    """Write one simple, non-runnable POTCAR placeholder."""
    missing = [element for element in element_order if element not in POTCAR_LABELS]
    if missing:
        raise KeyError("Add POTCAR labels for: " + ", ".join(sorted(missing)))

    potcar_labels = [POTCAR_LABELS[element] for element in element_order]
    path = directory / "POTCAR"
    path.write_text(
        "POTCAR NOT DISTRIBUTED\n\n"
        "Replace this placeholder with licensed PAW-PBE datasets in the "
        "following POSCAR order:\n"
        + " ".join(potcar_labels)
        + "\n",
        encoding="utf-8",
    )
    return path, potcar_labels


def vasp_system_label(
    surface_id: str,
    adsorbate_name: str | None = None,
    config_id: str | None = None,
) -> str:
    """Create labels such as ``mp-10260_111_OOH_config_10``."""
    if adsorbate_name is None:
        return f"{surface_id}_bare"

    config_number = None
    if config_id is not None:
        match = re.search(r"(\d+)$", str(config_id))
        if match:
            config_number = int(match.group(1))

    if config_number is None:
        suffix = str(config_id or "selected").replace(" ", "_")
        return f"{surface_id}_{adsorbate_name}_{suffix}"

    return f"{surface_id}_{adsorbate_name}_config_{config_number}"


def export_vasp_calculation(
    *,
    atoms: Atoms,
    directory: Path,
    system_label: str,
    metadata: dict,
    n_adsorbate_atoms: int = 0,
) -> dict:
    """Export one repository-safe AQCat RPBE static VASP folder."""
    if directory.exists():
        shutil.rmtree(directory)
    directory.mkdir(parents=True, exist_ok=True)

    poscar_path = directory / "POSCAR"
    ordering = write_poscar_with_selective_dynamics(
        poscar_path,
        atoms,
        system_label,
        n_adsorbate_atoms=n_adsorbate_atoms,
    )
    kpoint_metadata = write_kpoints(directory, atoms)
    incar_path, spin_metadata = write_incar(
        directory,
        system_label,
        ordering["ordered_symbols"],
        ordering["ordered_roles"],
    )
    job_path = write_job_script(
        directory,
        f"{system_label}_AQCat_RPBE_SP",
    )
    potcar_path, potcar_labels = write_potcar_placeholder(
        directory,
        ordering["element_order"],
    )

    case_metadata = {
        **metadata,
        "system_label": system_label,
        "source_frame": -1,
        "formula": atoms.get_chemical_formula(),
        "n_atoms": len(atoms),
        "n_adsorbate_atoms": n_adsorbate_atoms,
        "element_order": ordering["element_order"],
        "element_counts": ordering["counts"],
        "ordered_symbols": ordering["ordered_symbols"],
        "ordered_roles": ordering["ordered_roles"],
        "grouped_indices": ordering["grouped_indices"],
        "potcar_labels": potcar_labels,
        "kpoint_metadata": kpoint_metadata,
        "spin_metadata": spin_metadata,
        "vasp_run_mode": VASP_RUN_MODE,
        "potcar_distributed": False,
    }

    metadata_path = directory / "metadata.json"
    metadata_path.write_text(
        json.dumps(case_metadata, indent=2, default=str) + "\n",
        encoding="utf-8",
    )

    return {
        "system_label": system_label,
        "directory": str(directory),
        "formula": atoms.get_chemical_formula(),
        "n_atoms": len(atoms),
        "n_adsorbate_atoms": n_adsorbate_atoms,
        "element_order": " ".join(ordering["element_order"]),
        "potcar_labels": " ".join(potcar_labels),
        "spin_polarized": spin_metadata["spin_polarized"],
        "ISPIN": spin_metadata["ISPIN"],
        "MAGMOM": spin_metadata["MAGMOM_expression"],
        "magnetic_elements_present": " ".join(
            spin_metadata["magnetic_elements_present"]
        ),
        "kpoints": " ".join(map(str, kpoint_metadata["mesh"])),
        "a_length_A": kpoint_metadata["a_length_A"],
        "b_length_A": kpoint_metadata["b_length_A"],
        "kpoint_density_parameter": KPOINT_DENSITY,
        "POSCAR": str(poscar_path),
        "INCAR": str(incar_path),
        "KPOINTS": kpoint_metadata["path"],
        "job.sh": str(job_path),
        "POTCAR": str(potcar_path),
        "metadata.json": str(metadata_path),
    }


In [21]:
vasp_rows = []

dft_protocol = {
    "run_mode": VASP_RUN_MODE,
    "functional": "RPBE",
    "GGA": "RP",
    "dispersion": "none",
    "PREC": VASP_PREC,
    "ENCUT_eV": VASP_ENCUT_EV,
    "EDIFF_eV": VASP_EDIFF,
    "NELM": VASP_NELM,
    "ALGO": "Normal",
    "ISMEAR": 0,
    "SIGMA_eV": VASP_SIGMA_EV,
    "IBRION": -1,
    "NSW": 0,
    "ISIF": "omitted for static single point",
    "POTIM": "omitted for static single point",
    "EDIFFG": "omitted for static single point",
    "ISYM": 0,
    "SYMPREC": VASP_SYMPREC,
    "LREAL": "Auto",
    "ISTART": VASP_ISTART,
    "LORBIT": 11,
    "LWAVE": True,
    "LCHARG": False,
    "NPAR": VASP_NPAR,
    "NCORE": "not used",
    "KPAR": "not used",
    "dipole_correction": False,
    "spin_rule": (
        "ISPIN=2 when the slab contains Ce, Co, Cr, Cu, Fe, Mn, Mo, Ni, "
        "Os, Ru, V, or W; otherwise ISPIN=1"
    ),
    "initial_magnetic_moments_uB": AQCAT_INITIAL_MAGMOM,
    "nonmagnetic_slab_moment_uB": 0.0,
    "adsorbate_O_H_moment_uB": 0.0,
    "magmom_order": "exact POSCAR atom order",
    "poscar_order": "slab species first, then adsorbate O and H",
    "kpoint_scheme": "Monkhorst-Pack",
    "kpoint_density_parameter": KPOINT_DENSITY,
    "kpoint_rule": "round(40/|a|), round(40/|b|), 1",
}

vasp_rows.append(
    export_vasp_calculation(
        atoms=clean_final,
        directory=VASP_DIR / "bare",
        system_label=vasp_system_label(selected_surface),
        n_adsorbate_atoms=0,
        metadata={
            "surface": selected_surface,
            "bulk_id": BULK_ID,
            "miller_index": MILLER_INDEX,
            "surface_index": SURFACE_INDEX,
            "structure_type": "bare_slab",
            "model_name": MODEL_NAME,
            "aqcat_context": {
                "is_spin_off": IS_SPIN_OFF,
                "is_low_fi": IS_LOW_FIDELITY,
            },
            "source_trajectory": str(clean_trajectory_path),
            "AQCat_bare_model_energy_eV": clean_model_energy_eV,
            "energy_interpretation": (
                "Diagnostic bare-slab model energy; used for geometry "
                "optimization only and not subtracted from adsorption energies."
            ),
            "final_fmax_eV_A": clean_relaxation["final_fmax_eV_A"],
            "dft_protocol": dft_protocol,
        },
    )
)

available_adsorbates = set(
    global_minima.get("adsorbate", pd.Series(dtype=str)).tolist()
)
missing_adsorbates = set(ADSORBATE_NAMES) - available_adsorbates
if missing_adsorbates:
    print(
        "[WARN] No accepted global minimum was available for:",
        ", ".join(sorted(missing_adsorbates)),
    )

for adsorbate_name in ADSORBATE_NAMES:
    if global_minima.empty:
        continue

    selected = global_minima[
        global_minima["adsorbate"] == adsorbate_name
    ]
    if selected.empty:
        continue

    row = selected.iloc[0]
    final = aseio.read(str(row["trajectory"]), index=-1)
    n_adsorbate_atoms = ADSORBATE_SIZES[adsorbate_name]

    vasp_rows.append(
        export_vasp_calculation(
            atoms=final,
            directory=VASP_DIR / adsorbate_name,
            system_label=vasp_system_label(
                selected_surface,
                adsorbate_name,
                str(row["config_id"]),
            ),
            n_adsorbate_atoms=n_adsorbate_atoms,
            metadata={
                "surface": selected_surface,
                "bulk_id": BULK_ID,
                "miller_index": MILLER_INDEX,
                "surface_index": SURFACE_INDEX,
                "structure_type": "adsorbate_global_minimum",
                "adsorbate": adsorbate_name,
                "config_id": str(row["config_id"]),
                "placement_kind": str(row["placement_kind"]),
                "site_type": str(row["site_type"]),
                "site_family": str(row["site_family"]),
                "site_instance_key": str(row["site_instance_key"]),
                "neighbor_indices": str(row["neighbor_indices"]),
                "neighbor_composition": str(row["neighbor_composition"]),
                "model_name": MODEL_NAME,
                "aqcat_context": {
                    "is_spin_off": IS_SPIN_OFF,
                    "is_low_fi": IS_LOW_FIDELITY,
                },
                "source_trajectory": str(row["trajectory"]),
                "AQCat_pred_ads_energy_eV": float(
                    row["AQCat_pred_ads_energy_eV"]
                ),
                "final_fmax_eV_A": float(row["final_fmax_eV_A"]),
                "energy_interpretation": (
                    "Direct AQCat25-EV2 adsorption-energy prediction; no "
                    "bare-slab or molecular-reference subtraction."
                ),
                "dft_protocol": dft_protocol,
            },
        )
    )

vasp_manifest = pd.DataFrame(vasp_rows)
vasp_manifest.to_csv(VASP_DIR / "manifest.csv", index=False)
display(vasp_manifest)


,system_label,directory,formula,n_atoms,n_adsorbate_atoms,element_order,potcar_labels,spin_polarized,ISPIN,MAGMOM,...,kpoints,a_length_A,b_length_A,kpoint_density_parameter,POSCAR,INCAR,KPOINTS,job.sh,POTCAR,metadata.json
0,mp-10260_111_bare,/content/stageII_aqcat25_ev2_results/mp-10260_...,Ni36Sb12,48,0,Ni Sb,Ni Sb,True,2,36*5.00 12*0.00,...,5 5 1,8.466487,8.466487,40.0,/content/stageII_aqcat25_ev2_results/mp-10260_...,/content/stageII_aqcat25_ev2_results/mp-10260_...,/content/stageII_aqcat25_ev2_results/mp-10260_...,/content/stageII_aqcat25_ev2_results/mp-10260_...,/content/stageII_aqcat25_ev2_results/mp-10260_...,/content/stageII_aqcat25_ev2_results/mp-10260_...
1,mp-10260_111_O_config_9,/content/stageII_aqcat25_ev2_results/mp-10260_...,Ni36OSb12,49,1,Ni Sb O,Ni Sb O,True,2,36*5.00 13*0.00,...,5 5 1,8.466487,8.466487,40.0,/content/stageII_aqcat25_ev2_results/mp-10260_...,/content/stageII_aqcat25_ev2_results/mp-10260_...,/content/stageII_aqcat25_ev2_results/mp-10260_...,/content/stageII_aqcat25_ev2_results/mp-10260_...,/content/stageII_aqcat25_ev2_results/mp-10260_...,/content/stageII_aqcat25_ev2_results/mp-10260_...
2,mp-10260_111_OH_config_25,/content/stageII_aqcat25_ev2_results/mp-10260_...,HNi36OSb12,50,2,Ni Sb O H,Ni Sb O H,True,2,36*5.00 14*0.00,...,5 5 1,8.466487,8.466487,40.0,/content/stageII_aqcat25_ev2_results/mp-10260_...,/content/stageII_aqcat25_ev2_results/mp-10260_...,/content/stageII_aqcat25_ev2_results/mp-10260_...,/content/stageII_aqcat25_ev2_results/mp-10260_...,/content/stageII_aqcat25_ev2_results/mp-10260_...,/content/stageII_aqcat25_ev2_results/mp-10260_...
3,mp-10260_111_OOH_config_7,/content/stageII_aqcat25_ev2_results/mp-10260_...,HNi36O2Sb12,51,3,Ni Sb O H,Ni Sb O H,True,2,36*5.00 15*0.00,...,5 5 1,8.466487,8.466487,40.0,/content/stageII_aqcat25_ev2_results/mp-10260_...,/content/stageII_aqcat25_ev2_results/mp-10260_...,/content/stageII_aqcat25_ev2_results/mp-10260_...,/content/stageII_aqcat25_ev2_results/mp-10260_...,/content/stageII_aqcat25_ev2_results/mp-10260_...,/content/stageII_aqcat25_ev2_results/mp-10260_...


In [22]:
def parse_incar_tags(path: Path) -> dict[str, str]:
    """Read simple ``TAG = value`` lines from an INCAR."""
    tags: dict[str, str] = {}
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.split("#", 1)[0].strip()
        if not line or "=" not in line:
            continue
        key, value = line.split("=", 1)
        tags[key.strip().upper()] = value.strip()
    return tags


def expanded_magmom_count(expression: str) -> int:
    """Count moments represented by a compressed VASP MAGMOM expression."""
    total = 0
    for token in expression.split():
        if "*" in token:
            count_text, _ = token.split("*", 1)
            total += int(count_text)
        else:
            total += 1
    return total


required_static_tags = {
    "GGA": "RP",
    "PREC": VASP_PREC,
    "ENCUT": str(VASP_ENCUT_EV),
    "EDIFF": f"{VASP_EDIFF:.0E}",
    "NELM": str(VASP_NELM),
    "ALGO": "Normal",
    "ISMEAR": "0",
    "SIGMA": str(VASP_SIGMA_EV),
    "ISYM": "0",
    "SYMPREC": f"{VASP_SYMPREC:.0E}",
    "LREAL": "Auto",
    "IBRION": "-1",
    "NSW": "0",
    "NPAR": str(VASP_NPAR),
}
forbidden_static_tags = {
    "IVDW", "VDW_VERSION", "LDIPOL", "IDIPOL", "DIPOL",
    "ISIF", "POTIM", "EDIFFG", "NCORE", "KPAR",
}

validation_rows = []
for row in vasp_manifest.itertuples(index=False):
    incar_path = Path(row.INCAR)
    tags = parse_incar_tags(incar_path)

    errors = []
    for key, expected in required_static_tags.items():
        if tags.get(key) != expected:
            errors.append(f"{key}: expected {expected}, found {tags.get(key)}")

    present_forbidden = sorted(forbidden_static_tags & set(tags))
    if present_forbidden:
        errors.append("forbidden tags: " + ", ".join(present_forbidden))

    n_atoms = int(row.n_atoms)
    if int(row.ISPIN) == 2:
        expression = str(row.MAGMOM)
        if not expression:
            errors.append("ISPIN=2 but MAGMOM is missing")
        elif expanded_magmom_count(expression) != n_atoms:
            errors.append(
                "MAGMOM count does not match POSCAR atom count: "
                f"{expanded_magmom_count(expression)} != {n_atoms}"
            )
    elif "MAGMOM" in tags:
        errors.append("ISPIN=1 but MAGMOM was written")

    validation_rows.append(
        {
            "system_label": row.system_label,
            "ISPIN": row.ISPIN,
            "MAGMOM": row.MAGMOM,
            "n_atoms": n_atoms,
            "status": "PASS" if not errors else "FAIL",
            "details": "; ".join(errors),
        }
    )

vasp_input_validation = pd.DataFrame(validation_rows)
display(vasp_input_validation)

if not vasp_input_validation.empty and not (
    vasp_input_validation["status"] == "PASS"
).all():
    raise RuntimeError("At least one exported VASP folder failed validation.")


,system_label,ISPIN,MAGMOM,n_atoms,status,details
0,mp-10260_111_bare,2,36*5.00 12*0.00,48,PASS,
1,mp-10260_111_O_config_9,2,36*5.00 13*0.00,49,PASS,
2,mp-10260_111_OH_config_25,2,36*5.00 14*0.00,50,PASS,
3,mp-10260_111_OOH_config_7,2,36*5.00 15*0.00,51,PASS,


## 17. Export tables and reproducibility archive

The archive contains the complete trajectory history, validation tables,
site-analysis tables, selected POSCARs, and VASP single-point folders. The gated
AQCat checkpoint, patched source files, and licensed VASP potentials are not
included.


In [23]:
def export_selected_poscars(
    table: pd.DataFrame,
    category: str,
) -> pd.DataFrame:
    exported = []

    if table.empty:
        return pd.DataFrame()

    for row in table.itertuples(index=False):
        final = aseio.read(row.trajectory, index=-1)

        if category == "unique_site_minima":
            folder = f"{row.config_id}__{row.site_instance_key}"
        else:
            folder = f"{row.adsorbate}_global_min__{row.config_id}"

        folder = folder.replace("/", "_")
        path = write_poscar(
            final,
            POSCAR_DIR / category / row.adsorbate / folder,
        )

        exported.append(
            {
                "category": category,
                "adsorbate": row.adsorbate,
                "config_id": row.config_id,
                "site_type": row.site_type,
                "site_family": row.site_family,
                "site_instance_key": row.site_instance_key,
                "AQCat_pred_ads_energy_eV": (
                    row.AQCat_pred_ads_energy_eV
                ),
                "spin_context": row.spin_context,
                "trajectory": row.trajectory,
                "poscar": str(path),
            }
        )

    return pd.DataFrame(exported)


selected_poscars = pd.concat(
    [
        export_selected_poscars(
            unique_site_minima,
            "unique_site_minima",
        ),
        export_selected_poscars(
            global_minima,
            "global_minima",
        ),
    ],
    ignore_index=True,
)

tables = {
    "adsorbate_summary.csv": adsorbate_summary,
    "placement_summary.csv": placement_summary,
    "initial_configuration_metadata.csv": configuration_table,
    "clean_relaxation.csv": clean_relaxation_table,
    "relaxation_status.csv": relaxation_status,
    "trajectory_validation.csv": trajectory_validation,
    "all_adsorption_sites.csv": all_adsorption_sites,
    "unique_site_minima.csv": unique_site_minima,
    "site_family_minima.csv": site_family_minima,
    "global_minima.csv": global_minima,
    "aqcat_global_adsorption_energies.csv": global_adsorption_energies,
    "selected_poscars.csv": selected_poscars,
    "vasp_manifest.csv": vasp_manifest,
}

for filename, table in tables.items():
    table.to_csv(TABLE_DIR / filename, index=False)

run_metadata = {
    "workflow_stage": (
        "Stage II AQCat25-EV2 spin-aware adsorption-energy relaxation, "
        "trajectory validation, site analysis, and DFT input generation"
    ),
    "surface": selected_surface,
    "bulk_id": BULK_ID,
    "miller_index": MILLER_INDEX,
    "surface_index": SURFACE_INDEX,
    "model_name": MODEL_NAME,
    "checkpoint": CHECKPOINT_PATH.name,
    "random_sites_per_adsorbate": RANDOM_SITES_PER_ADSORBATE,
    "fmax_threshold_eV_A": FMAX_THRESHOLD,
    "max_relax_steps": MAX_RELAX_STEPS,
    "adsorbates": list(ADSORBATE_NAMES),
    "aqcat_context": {
        "spin_context": SPIN_CONTEXT,
        "is_spin_off": IS_SPIN_OFF,
        "is_low_fi": IS_LOW_FIDELITY,
    },
    "energy_interpretation": (
        "For adsorbate-slab systems, AQCat25-EV2 returns the referenced "
        "adsorption-energy target directly. AQCat_pred_ads_energy_eV equals "
        "the final model energy. The bare-slab model energy is used only for "
        "geometry optimization and is not subtracted."
    ),
    "bare_slab_model_energy_eV": clean_model_energy_eV,
    "vasp_protocol": dft_protocol,
    "potcar_distributed": False,
    "model_files_distributed": False,
}

(SYSTEM_DIR / "run_metadata.json").write_text(
    json.dumps(run_metadata, indent=2, default=str) + "\n",
    encoding="utf-8",
)

readme_text = f"""# Stage II AQCat25-EV2 reproducibility results

Surface: `{selected_surface}`
Model: `{MODEL_NAME}`
Context: high fidelity, spin on
Force threshold: `{FMAX_THRESHOLD} eV/Å`

## Scientific scope

This directory contains the one-slab public demonstration of the Stage II
AQCat25-EV2 branch used for the 24 top candidates selected after Stage I.

The bare slab is optimized first. O*, OH*, and OOH* are then placed on the
AQCat-relaxed slab and relaxed using the same high-fidelity, spin-on model
context. For adsorbate-slab systems, the final model output is stored directly
as `AQCat_pred_ads_energy_eV`.

No bare-slab subtraction, H2/H2O CHE referencing, generic ZPE, entropy,
solvation, limiting-potential, or overpotential correction is applied here.

## Main tables

- `tables/clean_relaxation.csv`
- `tables/relaxation_status.csv`
- `tables/trajectory_validation.csv`
- `tables/all_adsorption_sites.csv`
- `tables/unique_site_minima.csv`
- `tables/site_family_minima.csv`
- `tables/global_minima.csv`
- `tables/aqcat_global_adsorption_energies.csv`
- `vasp/manifest.csv`

## VASP directories

- `vasp/bare`
- `vasp/O`
- `vasp/OH`
- `vasp/OOH`

The VASP inputs use AQCat-style high-fidelity RPBE electronic settings with a
500 eV cutoff and Gaussian smearing. Spin polarization is enabled when the
system contains one of the AQCat25 magnetic elements. The folders are prepared
for single-point calculations with `IBRION = -1` and `NSW = 1`.

Every `POTCAR` is an intentional non-runnable placeholder. Replace it privately
with licensed PAW-PBE datasets in the stated element order.
"""

(SYSTEM_DIR / "README.md").write_text(
    textwrap.dedent(readme_text),
    encoding="utf-8",
)

archive_path = shutil.make_archive(
    base_name=str(ROOT_DIR / f"{SYSTEM_ID}_stageII"),
    format="zip",
    root_dir=str(ROOT_DIR),
    base_dir=SYSTEM_ID,
)

print("Result directory:", SYSTEM_DIR)
print("ZIP archive:", archive_path)
print("\nGenerated VASP meshes and spin settings:")
display(
    vasp_manifest[
        [
            "system_label",
            "spin_polarized",
            "ISPIN",
            "magnetic_elements_present",
            "kpoints",
        ]
    ]
)


Result directory: /content/stageII_aqcat25_ev2_results/mp-10260_111_term0_aqcat25_spin_on
ZIP archive: /content/stageII_aqcat25_ev2_results/mp-10260_111_term0_aqcat25_spin_on_stageII.zip

Generated VASP meshes and spin settings:


,system_label,spin_polarized,ISPIN,magnetic_elements_present,kpoints
0,mp-10260_111_bare,True,2,Ni,5 5 1
1,mp-10260_111_O_config_9,True,2,Ni,5 5 1
2,mp-10260_111_OH_config_25,True,2,Ni,5 5 1
3,mp-10260_111_OOH_config_7,True,2,Ni,5 5 1


In [24]:
import sys

if "google.colab" in sys.modules:
    from google.colab import files
    files.download(archive_path)
else:
    print("Archive available at:", archive_path)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Reproducibility notes and workflow connection

The principal output is `AQCat_pred_ads_energy_eV`. For every accepted O*, OH*,
or OOH* configuration,

$$
E_{\mathrm{ads}}^{\mathrm{AQCat}}
=E_{\mathrm{model}}^{\mathrm{final}}.
$$

This direct interpretation differs from the preceding eSEN/OC25 notebook, where
independently predicted total energies were combined with a CHE reference
subtraction. In the AQCat branch, the adsorption-energy reference is part of the
model target convention.

This notebook is the public demonstration of the **second Stage II branch**.
The full study repeated the same procedure for the 24 top candidates identified
in Stage I; one slab is selected here to keep the tutorial executable and easy
to audit.

The exported bare, O*, OH*, and OOH* structures are prepared for RPBE static
single-point DFT validation using the AQCat25 slab electronic settings. The
folders contain no empirical dispersion correction and no dipole-correction
block. For magnetic slabs, the atom-resolved initial moments follow AQCat25
Table 6 in the exact reordered POSCAR sequence, with adsorbate O/H initialized
to `0.00 μB`.

AQCat25-EV2 uses a binary global spin context rather than explicit atom-resolved
magnetic order. The generated ferromagnetic-style initialization reproduces the
AQCat high-throughput convention, but complex antiferromagnetic candidates may
require additional DFT calculations with alternative magnetic orderings.


## References

1. Allam, O.; Wander, B.; Kim, S.; et al. **AQCat25: Unlocking Spin-Aware,
   High-Fidelity Machine Learning Potentials for Heterogeneous Catalysis.**
   *npj Computational Materials* **2026**, *12*, 226.
   DOI: `10.1038/s41524-026-02099-6`.

2. AQCat25-EV2 model card, checkpoint, FiLM patch, and inference instructions:
   `https://huggingface.co/SandboxAQ/aqcat25-ev2`.

3. Lan, J.; Palizhati, A.; Shuaibi, M.; et al. **AdsorbML: A Leap in
   Efficiency for Adsorption Energy Calculations Using Generalizable Machine
   Learning Potentials.** *npj Computational Materials* **2023**, *9*, 172.

4. Chanussot, L.; Das, A.; Goyal, S.; et al. **The Open Catalyst 2020 Dataset
   and Community Challenges.** *ACS Catalysis* **2021**, *11*, 6059-6072.
